# ISU Campus Assistant — a RAG demo

Ask questions about **ISU courses, billing, housing and dining** and get answers grounded in the
university's own catalog and websites, with links back to every source page.

This notebook is the [ISU Course Catalog Collector](https://catalog.illinoisstate.edu) plus four more layers:

| Layer | What it does |
|---|---|
| **1. Collect** | Pulls all ~4,800 courses from the catalog's Coursedog JSON API, then crawls the billing, housing and dining sites. |
| **2. Sync** | Every launch it re-fetches, fingerprints each course, and reports what was **added / removed / updated** since last run. |
| **3. Index** | Courses stay one chunk each; web pages are split on their headings into ~1,000-character chunks. All of it is embedded with MiniLM into one **Chroma** collection, tagged by section, alongside a BM25 keyword index. |
| **4. Answer** | Gemini reads only the retrieved courses and answers with citations. |
| **5. Interface** | ReggieBot, a Gradio chat app in Illinois State red and white, with starter questions and source links under every answer. |
| **6. Publish** | The corpus and vector DB push to a Hugging Face dataset repo so they outlive the Colab runtime. |

### Before you run it

1. Get a free API key at **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**.
2. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret** →
   name it `GOOGLE_API_KEY`, paste the key, and switch **Notebook access** on.
3. **First time only:** tick `REFRESH_DATA` in the crawler cell (§8) and run it. It fetches the
   catalog and builds the vector database, about two minutes.
4. `Runtime → Run all` any time after that. The crawler cell is skipped while `REFRESH_DATA` is
   unticked, and the assistant starts from the saved data in seconds.

> No key? The app still runs — it falls back to retrieval-only mode and shows the matching
> courses without the written answer.

---
## 0 · Install

`google-genai` for the answers, `gradio` for the interface, `rank-bm25` for keyword search,
`sentence-transformers` for the embeddings, `chromadb` for the vector database, and
`huggingface_hub` to publish the finished store.

In [ ]:
%pip install -q google-genai gradio rank-bm25 sentence-transformers chromadb huggingface_hub trafilatura beautifulsoup4 lxml
print("done - if Colab asks you to restart the runtime, do it and then Run all again.")

---
## 1 · Configuration

Nothing here is secret. Every value below is visible in the catalog website's own network traffic.

In [ ]:
from pathlib import Path

# --- Catalog source ----------------------------------------------------------
API_BASE   = "https://app.coursedog.com/api/v1"
SCHOOL_ID  = "illinoisstate_peoplesoft_direct"   # ISU's Coursedog tenant
SITE       = "https://catalog.illinoisstate.edu" # also used to build course links

CATALOG_ID = None    # None = auto-pick the catalog that is live today
PAGE_SIZE  = 200     # courses per request
PAUSE      = 0.15    # seconds between requests - be a polite client
TIMEOUT    = 30
MAX_RETRIES = 4

# --- Where everything is saved -----------------------------------------------
# The cache holds the catalog snapshot (for change detection), the document
# corpus, the Chroma vector database, and the experiment results.
#
#   "drive"   Save everything to DRIVE_FOLDER in your Google Drive.   <- default
#   "off"     Colab's local disk only. Fast, and wiped when the runtime ends.
#
# On Drive, Chroma's database is opened from a local working copy and copied
# back to DRIVE_FOLDER/chroma after every change. Chroma keeps its index in
# SQLite, which needs file locking that Google Drive's mount does not reliably
# provide, so a database written live on Drive can fail with "database is
# locked" or "disk I/O error". Copying a closed database is safe.
DRIVE_MODE   = "drive"
DRIVE_FOLDER = "/content/drive/MyDrive/isu_catalog_cache"

CACHE_DIR = Path("isu_catalog_cache")
DRIVE_PATH = None

if DRIVE_MODE in ("drive", "direct", "mirror"):       # "direct" / "mirror" = older names
    try:
        from google.colab import drive
    except ImportError:
        print("not running in Colab - saving to the local folder instead of Google Drive")
    else:
        drive.mount("/content/drive")
        DRIVE_PATH = Path(DRIVE_FOLDER)
        DRIVE_PATH.mkdir(parents=True, exist_ok=True)
        if DRIVE_MODE != "mirror":
            # A cache an earlier run left on local disk moves to Drive, once.
            if (CACHE_DIR / "corpus.jsonl").exists() and not (DRIVE_PATH / "corpus.jsonl").exists():
                import shutil
                shutil.copytree(CACHE_DIR, DRIVE_PATH, dirs_exist_ok=True)
                print(f"copied the existing local cache into {DRIVE_PATH}")
            CACHE_DIR = DRIVE_PATH

CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- Vector database ---------------------------------------------------------
CHROMA_DIR      = CACHE_DIR / "chroma"       # the saved Chroma store
COLLECTION_NAME = "isu_courses"
# Where Chroma is opened: a local working copy when the cache is on Drive.
CHROMA_WORK     = Path("chroma_working_copy") if CACHE_DIR == DRIVE_PATH else CHROMA_DIR

# --- Publishing (Hugging Face Hub) -------------------------------------------
# Set this to "<your-hf-username>/isu-course-catalog-rag" and add an HF_TOKEN
# Colab secret to push the corpus and vector DB to a dataset repo.
HF_DATASET_REPO = None
HF_PRIVATE      = False

# --- Retrieval + answering ---------------------------------------------------
EMBED_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"  # small, fast, runs on CPU
GEMINI_MODEL  = None     # None = auto-pick the best flash model your key can see
PROBE_MODELS  = True     # test each model with a tiny call at startup and drop dead ones
TOP_K         = 8        # courses handed to the model as context
SHOW_SOURCES  = 5        # source cards shown under an answer
SHARE_LINK    = True     # True = Gradio prints a public https://...gradio.live link

# --- Illinois State University brand colors ----------------------------------
ISU_RED   = "#CC0000"
ISU_WHITE = "#FFFFFF"
ISU_BLACK = "#000000"
ISU_BLUE  = "#56758F"    # official secondary, used sparingly for links

print("cache:", CACHE_DIR.resolve())

---
## 2 · Collect — read the catalog from its own API

`catalog.illinoisstate.edu` is a Coursedog single-page app: the HTML is an empty shell and the
course list is injected by JavaScript. That JavaScript calls a public JSON endpoint, and this is
that endpoint. It answers only when the request carries `Origin`/`Referer` headers naming the
catalog site — that is the whole trick, and it is why the session below sets them once.

In [ ]:
import hashlib
import html as html_lib
import json
import os
import re
import time
from collections import Counter
from datetime import date, datetime

import requests

SESSION = requests.Session()
SESSION.headers.update({
    "Content-Type": "application/json",
    # Coursedog hosts many schools on one API and answers only when the request
    # says which catalog site it is for. Without these two headers: 401.
    "Origin": SITE,
    "Referer": SITE + "/",
    # Identify the client honestly instead of pretending to be a browser.
    "User-Agent": "ISU-catalog-rag/1.0 (coursework; contact: your-email@ilstu.edu)",
})


def request_json(method, url, **kwargs):
    """One HTTP call. Retries what is worth retrying, explains what is not."""
    delay = 1.0
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = SESSION.request(method, url, timeout=TIMEOUT, **kwargs)
        except requests.RequestException as exc:          # timeout, DNS, dropped connection
            if attempt == MAX_RETRIES:
                raise
            time.sleep(delay)
            delay *= 2
            continue

        if resp.status_code in (401, 403):
            raise PermissionError(
                f"HTTP {resp.status_code} from {url.split('?')[0]}\n"
                f"The API rejected the request. Check that SESSION.headers still sends\n"
                f"Origin/Referer = {SITE} - that is what identifies which catalog to serve."
            )

        if resp.status_code == 429 or resp.status_code >= 500:
            if attempt == MAX_RETRIES:
                resp.raise_for_status()
            time.sleep(delay)
            delay *= 2
            continue

        resp.raise_for_status()
        return resp.json()


# --- which catalog is live today ---------------------------------------------

def list_catalogs():
    return request_json("GET", f"{API_BASE}/ca/{SCHOOL_ID}/catalogs")


def pick_current_catalog(catalogs):
    """Newest non-archived catalog whose effective start date has already passed."""
    today = date.today().isoformat()
    live = [c for c in catalogs
            if not c.get("archivedAt")
            and (c.get("effectiveStartDate") or "9999") <= today]
    pool = live or [c for c in catalogs if not c.get("archivedAt")] or catalogs
    return max(pool, key=lambda c: c.get("effectiveStartDate") or "")


# --- every course in that catalog --------------------------------------------

COLUMNS = ",".join([
    "customFields.rawCourseId", "code", "name", "longName", "courseNumber",
    "subjectCode", "description", "career", "college", "status", "credits",
    "departments", "attributes", "courseGroupId",
])

# Same filter the catalog website sends: everything except Inactive courses.
COURSE_FILTER = {
    "condition": "AND",
    "filters": [{
        "condition": "and",
        "id": "status-course-group",
        "filters": [{
            "id": "status-course", "condition": "field", "name": "status",
            "inputType": "select", "group": "course", "type": "isNot",
            "value": "Inactive", "customField": False,
        }],
    }],
}


def fetch_courses(catalog_id, effective, page_size=PAGE_SIZE, progress=None):
    """Return every course record in the catalog, one page at a time."""
    url = f"{API_BASE}/cm/{SCHOOL_ID}/courses/search/%24filters"
    courses, seen, skip, total = [], set(), 0, None

    while total is None or skip < total:
        params = {
            "catalogId": catalog_id,
            "skip": skip,
            "limit": page_size,
            "orderBy": "code",
            "formatDependents": "false",
            "effectiveDatesRange": f"{effective},{effective}",
            "ignoreEffectiveDating": "false",
            "ignoreTotalCount": "false",
            "columns": COLUMNS,
        }
        payload = request_json("POST", url, params=params, json=COURSE_FILTER)

        if total is None:
            total = payload.get("listLength") or 0

        data = payload.get("data") or {}
        page = list(data.values()) if isinstance(data, dict) else list(data)
        if not page:
            break

        for course in page:
            key = course.get("_id") or course.get("courseGroupId")
            if key not in seen:
                seen.add(key)
                courses.append(course)

        skip += page_size
        if progress:
            progress(f"fetched {min(skip, total)} of {total} courses")
        time.sleep(PAUSE)

    return courses

### Clean the records, then turn each course into one retrievable document

A retriever embeds text, not JSON. `to_document()` writes each course as a small self-contained
page — header, facts, description — so a single chunk carries everything needed to answer about
that course, and the metadata stays attached for the source cards.

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")
WS_RE  = re.compile(r"[ \t ​]+")


def clean(text):
    """Strip stray HTML, unescape entities, collapse whitespace."""
    if not text:
        return ""
    text = TAG_RE.sub(" ", str(text))
    text = html_lib.unescape(text)
    text = text.replace(" ", " ").replace("\r", "")
    text = WS_RE.sub(" ", text)
    text = re.sub(r"\s+([,.;:!?)])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def credit_label(credits):
    """'3', '1-3', or '' - matches how the catalog displays credit hours."""
    if not isinstance(credits, dict):
        return ""
    hours = credits.get("creditHours") or {}
    low, high = hours.get("min"), hours.get("max")
    if low is None and high is None:
        n = credits.get("numberOfCredits")
        return "" if n is None else str(n)
    if low is None:
        low = high
    if high is None or high == low:
        return str(low)
    return f"{low}-{high}"


def split_code_name(value):
    """'ARTSC - College of Arts and Sciences' -> ('ARTSC', 'College of Arts...')."""
    value = clean(value)
    if " - " in value:
        code, _, name = value.partition(" - ")
        return code.strip(), name.strip()
    return "", value


def to_record(course, catalog_name):
    """Flatten one API record into the fields we actually care about."""
    college_code, college = split_code_name(course.get("college"))
    departments = [clean(d.get("name") or d.get("displayName"))
                   for d in (course.get("departments") or []) if isinstance(d, dict)]
    group_id = course.get("courseGroupId") or ""
    return {
        "code": clean(course.get("code")),
        "subject_code": clean(course.get("subjectCode")),
        "course_number": clean(course.get("courseNumber")),
        # longName is the full title; name is the abbreviated Peoplesoft one.
        "title": clean(course.get("longName") or course.get("name")),
        "short_title": clean(course.get("name")),
        "description": clean(course.get("description")),
        "credits": credit_label(course.get("credits")),
        "repeatable": bool((course.get("credits") or {}).get("repeatable")),
        "career": clean(course.get("career")),
        "college": college,
        "college_code": college_code,
        "departments": [d for d in departments if d],
        "attributes": [clean(a) for a in (course.get("attributes") or []) if a],
        "status": clean(course.get("status")),
        "course_group_id": group_id,
        "url": f"{SITE}/courses/{group_id}" if group_id else SITE + "/courses",
        "catalog": catalog_name,
        "section": "courses",      # which part of the university this came from
        "fetched_at": "",          # web pages carry a date here; courses do not
    }


def to_document(rec):
    """One self-contained chunk of text per course - what the retriever embeds."""
    header = f"{rec['code']} - {rec['title']}".strip(" -")
    facts = [f"Course code: {rec['code']}"]
    if rec["subject_code"]:
        facts.append(f"Subject: {rec['subject_code']}")
    if rec["credits"]:
        facts.append(f"Credit hours: {rec['credits']}")
    if rec["career"]:
        facts.append(f"Level: {rec['career']}")
    if rec["college"]:
        facts.append(f"College: {rec['college']}")
    if rec["departments"]:
        facts.append("Department: " + "; ".join(rec["departments"]))
    if rec["attributes"]:
        facts.append("Attributes: " + "; ".join(rec["attributes"]))
    facts.append(f"Catalog: {rec['catalog']}")
    facts.append(f"Source: {rec['url']}")

    body = rec["description"] or "No catalog description published for this course."
    return f"{header}\n" + "\n".join(facts) + f"\n\n{body}"


def build_documents(courses, catalog_name):
    """API records -> deduplicated, sorted, uniquely-identified documents."""
    records = [to_record(c, catalog_name) for c in courses]
    records = [r for r in records if r["code"]]

    seen, unique = set(), []
    for r in records:
        key = (r["code"], r["course_group_id"])
        if key not in seen:
            seen.add(key)
            unique.append(r)
    records = unique
    records.sort(key=lambda r: (r["subject_code"], r["course_number"], r["code"]))

    # A handful of course codes are reused by two different course records
    # (e.g. SOC 317), so fall back to the catalog's own id to keep doc ids unique.
    counts = Counter(r["code"] for r in records)

    def doc_id(rec):
        if counts[rec["code"]] > 1 and rec["course_group_id"]:
            return f"{rec['code']}-{rec['course_group_id']}"
        return rec["code"]

    documents = [{"id": doc_id(r), "text": to_document(r), "metadata": r} for r in records]
    assert len(set(d["id"] for d in documents)) == len(documents), "duplicate document ids"
    return documents


def collect_catalog(progress=None):
    """Fetch the live catalog and return (documents, catalog_meta)."""
    say = progress or (lambda msg: None)

    say("checking which catalog is live")
    catalogs = list_catalogs()
    catalog = next((c for c in catalogs if c["id"] == CATALOG_ID), None) or pick_current_catalog(catalogs)

    meta = {
        "catalog_id": catalog["id"],
        "catalog_name": catalog.get("displayName", ""),
        "effective": catalog.get("effectiveStartDate") or date.today().isoformat(),
        "checked_at": datetime.now().isoformat(timespec="seconds"),
    }

    say(f"reading {meta['catalog_name']}")
    courses = fetch_courses(meta["catalog_id"], meta["effective"], progress=say)
    documents = build_documents(courses, meta["catalog_name"])

    meta["course_count"] = len(documents)
    meta["subject_count"] = len({d["metadata"]["subject_code"] for d in documents})
    say(f"built {len(documents)} course documents")
    return documents, meta

---
## 3 · Crawl the rest of the website

The catalog arrives as clean JSON. Billing, housing and dining do not — they are ordinary web pages
spread across subdomains, so this section adds a second collector.

Three things make it different from the catalog collector:

* **Politeness is enforced, not assumed.** `robots.txt` is parsed per host and obeyed, the crawl
  stays inside an allowlist of ISU hosts, and there is a delay between requests.
* **Boilerplate has to go.** Nav bars and footers repeat on every page. Left in, they would dominate
  retrieval - every page would look identical to the retriever.
* **Pages need real chunking.** A billing page is long and covers several topics, so it is split on
  its headings into ~1,000-character chunks. Each chunk carries the page title, its heading, the URL
  and the date it was fetched, so a retrieved fragment can still be cited and still makes sense on
  its own.

Anything behind a login is out of scope by design. The assistant can explain how billing works and
where to pay; it can never see a student's balance.

In [ ]:
from urllib import robotparser
from urllib.parse import urljoin, urlparse, urldefrag

# --- what to crawl -----------------------------------------------------------
# Each section is a label plus the pages to start from. Add or remove freely.
WEB_SOURCES = {
    "billing": ["https://studentaccounts.illinoisstate.edu/"],
    "housing": ["https://housing.illinoisstate.edu/"],
    "dining":  ["https://dining.illinoisstate.edu/"],
}

ALLOWED_HOSTS  = {"illinoisstate.edu"}      # plus any subdomain of these
CRAWL_MAX_PAGES = 120                       # per section - raise once it works
CRAWL_MAX_DEPTH = 3                         # links away from the seed
CRAWL_PAUSE     = 0.5                       # seconds between requests
CRAWL_TIMEOUT   = 20
CHUNK_CHARS     = 1000                      # target characters per chunk
CHUNK_OVERLAP   = 150                       # characters repeated between chunks
MIN_CHUNK_CHARS = 120                       # below this a chunk is not worth keeping

SKIP_SUFFIXES = (".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx", ".zip",
                 ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".mp4", ".mov",
                 ".mp3", ".ics", ".xml", ".rss", ".json")

WEB_SESSION = requests.Session()
WEB_SESSION.headers.update({
    "User-Agent": "ISU-campus-rag/1.0 (student project; contact: your-email@ilstu.edu)",
    "Accept": "text/html,application/xhtml+xml",
})

_ROBOTS = {}


def robots_for(host):
    """One cached robots.txt parser per host."""
    if host not in _ROBOTS:
        parser = robotparser.RobotFileParser()
        parser.set_url(f"https://{host}/robots.txt")
        try:
            parser.read()
        except Exception:
            parser.parse([])                 # unreadable robots.txt = nothing disallowed
        _ROBOTS[host] = parser
    return _ROBOTS[host]


def host_allowed(host):
    return any(host == h or host.endswith("." + h) for h in ALLOWED_HOSTS)


def crawlable(url):
    """Host allowlist, file-type filter and robots.txt, in that order."""
    parts = urlparse(url)
    if parts.scheme not in ("http", "https") or not host_allowed(parts.netloc):
        return False
    if parts.path.lower().endswith(SKIP_SUFFIXES):
        return False
    if parts.query:                          # search pages and filters explode the crawl
        return False
    return robots_for(parts.netloc).can_fetch(WEB_SESSION.headers["User-Agent"], url)


def normalize(url):
    """Drop the #fragment and any trailing slash so one page has one id."""
    url, _ = urldefrag(url)
    return url.rstrip("/") or url


def within_seed(url, seeds):
    """Keep a section's crawl inside that section.

    Without this, one link back to the university home page pulls the whole of
    www.illinoisstate.edu into the "billing" section, mislabelled.
    """
    target = urlparse(url)
    target_path = target.path or "/"           # a bare host normalizes to an empty path
    for seed in seeds:
        base = urlparse(seed)
        if target.netloc != base.netloc:
            continue
        prefix = base.path.rstrip("/")         # "" when the seed IS the host root
        if not prefix or target_path.startswith(prefix):
            return True
    return False

### Turning a web page into text

Two extractors, tried in order. `trafilatura` is very good at throwing away navigation and keeping
the article; BeautifulSoup gives us the heading structure that the chunker splits on. The
BeautifulSoup walk runs first because the headings matter, and trafilatura is the safety net when a
page's markup defeats it.

In [ ]:
DROP_TAGS = ("script", "style", "nav", "header", "footer", "aside", "form",
             "noscript", "iframe", "svg", "button")

MAIN_HINTS = ("main", "article", "#main-content", "#content", ".main-content",
              ".page-content", "#primary", ".entry-content")


def extract_page(html, url):
    """(page title, [(heading, text), ...]) with navigation and chrome removed."""
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html, "lxml")
    title = clean(soup.title.get_text()) if soup.title else ""
    if not title and soup.find("h1"):
        title = clean(soup.find("h1").get_text())

    for tag in soup(list(DROP_TAGS)):        # boilerplate would otherwise dominate retrieval
        tag.decompose()

    main = None
    for hint in MAIN_HINTS:
        main = soup.select_one(hint)
        if main:
            break
    main = main or soup.body or soup

    blocks, heading, buffer = [], "", []

    def flush():
        text = clean(" ".join(buffer))
        if text:
            blocks.append((heading, text))
        buffer.clear()

    for node in main.find_all(["h1", "h2", "h3", "h4", "p", "li", "td", "th", "dd", "dt"]):
        text = clean(node.get_text(" "))
        if not text:
            continue
        if node.name in ("h1", "h2", "h3", "h4"):
            flush()                          # a heading ends the previous block
            heading = text
        else:
            buffer.append(text)
    flush()

    if sum(len(t) for _, t in blocks) < 200:
        blocks = trafilatura_blocks(html) or blocks     # markup defeated the walk above

    return title, blocks


def trafilatura_blocks(html):
    """Fallback extractor - no heading structure, but excellent boilerplate removal."""
    try:
        import trafilatura
    except ImportError:
        return []
    text = trafilatura.extract(html, include_comments=False, include_tables=True) or ""
    text = clean(text)
    return [("", text)] if text else []

### Chunking

This is the part the catalog did not need. Each `(heading, text)` block is split into pieces of
about `CHUNK_CHARS` characters on sentence boundaries, with `CHUNK_OVERLAP` characters repeated
between neighbours so a sentence straddling a boundary is not lost.

Every chunk is written in the same shape as a course document - a header line, a short facts block,
then the body - so BM25 and the embedding model see one consistent format across the whole corpus.

In [ ]:
SENTENCE_END = re.compile(r"(?<=[.!?])\s+")


def split_text(text, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    """Split on sentence boundaries into ~size-character pieces that overlap."""
    sentences = SENTENCE_END.split(text)
    chunks, current = [], ""

    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= size:
            current = f"{current} {sentence}".strip()
            continue
        if current:
            chunks.append(current)
        tail = current[-overlap:] if overlap and current else ""
        current = f"{tail} {sentence}".strip() if tail else sentence
        while len(current) > size * 1.6:     # one enormous sentence: hard-split it
            chunks.append(current[:size])
            current = current[size - overlap:]
    if current:
        chunks.append(current)
    return [c for c in chunks if len(c) >= MIN_CHUNK_CHARS] or ([text] if text else [])


def web_document(section, url, title, heading, body, position, fetched_at):
    """One chunk, written in the same shape as a course document."""
    label = " - ".join(p for p in (title, heading) if p) or url
    facts = [f"Section: {section.title()}"]
    if heading:
        facts.append(f"Page section: {heading}")
    facts.append(f"Source: {url}")
    facts.append(f"Last checked: {fetched_at[:10]}")

    text = f"{label}\n" + "\n".join(facts) + f"\n\n{body}"
    doc_id = f"{section}:{hashlib.sha1(url.encode()).hexdigest()[:10]}#{position}"

    return {
        "id": doc_id,
        "text": text,
        "metadata": {
            "section": section,              # what the UI filter and Chroma filter use
            "code": "",                      # web pages have no course code
            "title": title or url,
            "heading": heading,
            "description": body[:400],
            "url": url,
            "host": urlparse(url).netloc,
            "fetched_at": fetched_at,
            "subject_code": "", "course_number": "", "short_title": title,
            "credits": "", "career": "", "college": "", "college_code": "",
            "departments": [], "attributes": [], "status": "", "repeatable": False,
            "course_group_id": "", "catalog": "",
        },
    }

### The crawl itself

Breadth-first from each seed, one section at a time, stopping at `CRAWL_MAX_PAGES` or
`CRAWL_MAX_DEPTH`. Failures are counted and skipped rather than fatal - one dead link should not end
a crawl of several hundred pages.

In [ ]:
def crawl_section(section, seeds, progress=None):
    """Crawl one section of the site and return its chunked documents."""
    say = progress or (lambda msg: None)
    queue = [(normalize(u), 0) for u in seeds]
    seen, documents, failures = set(), [], 0

    while queue and len(seen) < CRAWL_MAX_PAGES:
        url, depth = queue.pop(0)
        if url in seen or not crawlable(url) or not within_seed(url, seeds):
            continue
        seen.add(url)

        try:
            resp = WEB_SESSION.get(url, timeout=CRAWL_TIMEOUT)
            resp.raise_for_status()
            if "html" not in resp.headers.get("Content-Type", ""):
                continue
        except Exception:
            failures += 1
            continue

        fetched_at = datetime.now().isoformat(timespec="seconds")
        title, blocks = extract_page(resp.text, url)

        position = 0
        for heading, text in blocks:
            for piece in split_text(text):
                documents.append(web_document(section, url, title, heading,
                                              piece, position, fetched_at))
                position += 1

        if depth < CRAWL_MAX_DEPTH:          # queue this page's links for the next level
            from bs4 import BeautifulSoup
            for link in BeautifulSoup(resp.text, "lxml").find_all("a", href=True):
                nxt = normalize(urljoin(url, link["href"]))
                if nxt not in seen and crawlable(nxt) and within_seed(nxt, seeds):
                    queue.append((nxt, depth + 1))

        if len(seen) % 10 == 0:
            say(f"{section}: {len(seen)} pages, {len(documents)} chunks")
        time.sleep(CRAWL_PAUSE)

    say(f"{section}: {len(seen)} pages -> {len(documents)} chunks"
        + (f" ({failures} failed)" if failures else ""))
    if not documents:                          # silence here would look like "no billing pages exist"
        print(f"[crawl] WARNING: {section} produced nothing from {seeds}. "
              f"Run check_sources() to see which step rejected it.")
    return documents


def check_sources(sources=None):
    """Fetch each seed once and print what the crawler will see. Run this before a full crawl."""
    print(f"{'section':<10} {'HTTP':<6} {'robots':<8} {'in seed':<9} {'text':<8} links  url")
    for section, seeds in (sources or WEB_SOURCES).items():
        for seed in seeds:
            url = parts = None
            try:
                url = normalize(seed)
                parts = urlparse(url)
                robots_ok = robots_for(parts.netloc).can_fetch(
                    WEB_SESSION.headers["User-Agent"], url)
                inside = within_seed(url, seeds)
                resp = WEB_SESSION.get(url, timeout=CRAWL_TIMEOUT)
                title, blocks = extract_page(resp.text, url)
                chars = sum(len(t) for _, t in blocks)
                from bs4 import BeautifulSoup
                links = sum(1 for a in BeautifulSoup(resp.text, "lxml").find_all("a", href=True)
                            if crawlable(normalize(urljoin(url, a["href"])))
                            and within_seed(normalize(urljoin(url, a["href"])), seeds))
                flag = "" if (robots_ok and inside and chars > 200) else "   <-- problem"
                print(f"{section:<10} {resp.status_code:<6} {str(robots_ok):<8} {str(inside):<9} "
                      f"{chars:<8} {links:<6} {url}{flag}")
            except Exception as exc:
                print(f"{section:<10} {'ERR':<6} {type(exc).__name__}: {str(exc)[:60]}  {seed}")


# Words that say which part of the university a question is about. Used when the
# dropdown is on "Everything" so 4,842 courses cannot drown 300 billing chunks.
SECTION_HINTS = {
    "billing": ["bill", "billing", "tuition", "payment", "pay", "paid", "due", "invoice",
                "refund", "balance", "late fee", "installment", "echeck", "e-check",
                "student account", "charge", "cost of attendance", "deposit"],
    "housing": ["housing", "dorm", "residence hall", "res hall", "roommate", "live on campus",
                "living on campus", "off campus", "off-campus", "room rate", "move-in", "move in",
                "suite", "apartment", "watterson", "tri-towers", "hewett", "manchester",
                "housing contract", "where do i live", "place to live"],
    "dining":  ["dining", "meal plan", "meal swipe", "flex dollar", "food", "eat", "menu",
                "cafeteria", "dining hall", "commons", "nutrition"],
    "courses": ["course", "class", "prerequisite", "credit hour", "syllabus", "gen ed",
                "general education", "major", "minor", "elective", "semester hours",
                "undergraduate course", "graduate course"],
}


# Whole words, with an optional plural. Plain substring matching finds "eat" inside
# "weather"; strict word boundaries then miss "meal plans" because the hint is
# singular. This does both: \b...(s|es)?\b.
HINT_PATTERNS = {
    section: re.compile(r"\b(?:" + "|".join(re.escape(w) for w in words) + r")(?:e?s)?\b")
    for section, words in SECTION_HINTS.items()
}


def route_sections(query):
    """Which sections a question is probably about. Empty set = no opinion."""
    text = (query or "").lower()
    hits = {section for section, pattern in HINT_PATTERNS.items() if pattern.search(text)}
    return hits - {"courses"} or hits          # a billing word beats a generic "class"


def collect_website(progress=None):
    """Crawl every configured section. Returns chunked documents for all of them."""
    say = progress or (lambda msg: None)
    documents = []
    for section, seeds in WEB_SOURCES.items():
        say(f"crawling {section}")
        documents.extend(crawl_section(section, seeds, progress=say))
    say(f"website: {len(documents)} chunks from {len(WEB_SOURCES)} sections")
    return documents

---
## 4 · Sync — notice when the catalog changes

Each course document gets a fingerprint (a hash of its text). The fingerprints from the previous
run are kept in `snapshot.json`, so comparing the two tells us exactly what moved:

* **added** — a course that did not exist last time
* **removed** — a course that is gone or newly Inactive
* **updated** — same course, edited description / credits / attributes / department

Two things use this. The chat header reports it to you, and the index uses it to re-embed
*only* the courses that actually changed instead of all 4,800.

In [ ]:
SNAPSHOT_PATH = CACHE_DIR / "snapshot.json"
CORPUS_PATH   = CACHE_DIR / "corpus.jsonl"


def fingerprint(doc):
    return hashlib.sha1(doc["text"].encode("utf-8")).hexdigest()[:16]


def load_snapshot():
    if SNAPSHOT_PATH.exists():
        try:
            return json.loads(SNAPSHOT_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"meta": {}, "fingerprints": {}}


def save_snapshot(documents, meta):
    SNAPSHOT_PATH.write_text(json.dumps({
        "meta": meta,
        "fingerprints": {d["id"]: fingerprint(d) for d in documents},
    }, indent=0), encoding="utf-8")

    with open(CORPUS_PATH, "w", encoding="utf-8") as f:     # the RAG corpus itself
        for doc in documents:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")


def mirror_to_drive():
    """Copy the finished cache to Drive (DRIVE_MODE = "mirror").

    The database is built on local disk and copied once it is closed, which
    avoids asking SQLite to hold locks on a FUSE mount that cannot provide them.
    """
    if DRIVE_MODE != "mirror" or DRIVE_PATH is None:
        return None

    import shutil
    copied = 0
    for item in CACHE_DIR.iterdir():
        target = DRIVE_PATH / item.name
        if item.is_dir():
            shutil.rmtree(target, ignore_errors=True)
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
        copied += 1
    print(f"mirrored {copied} item(s) to {DRIVE_PATH}")
    return DRIVE_PATH


def restore_from_drive():
    """Copy a previous mirror back, so a fresh runtime skips the rebuild."""
    if DRIVE_MODE != "mirror" or DRIVE_PATH is None or not DRIVE_PATH.exists():
        return None

    import shutil
    restored = 0
    for item in DRIVE_PATH.iterdir():
        target = CACHE_DIR / item.name
        if target.exists():
            continue                     # a local copy is the newer one - leave it
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
        restored += 1
    if restored:
        print(f"restored {restored} item(s) from {DRIVE_PATH}")
    return CACHE_DIR


def diff_documents(documents, previous):
    """What changed between the last snapshot and this fetch."""
    old = previous.get("fingerprints") or {}
    new = {d["id"]: fingerprint(d) for d in documents}

    if not old:
        return {"first_run": True, "added": [], "removed": [], "updated": [],
                "changed_ids": set(new)}

    added   = sorted(set(new) - set(old))
    removed = sorted(set(old) - set(new))
    updated = sorted(i for i in set(new) & set(old) if new[i] != old[i])
    return {"first_run": False, "added": added, "removed": removed, "updated": updated,
            "changed_ids": set(added) | set(updated)}


def describe_changes(changes):
    """One short sentence for the header strip."""
    if changes.get("first_run"):
        return "first run - baseline snapshot saved"
    bits = []
    for label, key in (("added", "added"), ("updated", "updated"), ("removed", "removed")):
        n = len(changes[key])
        if n:
            bits.append(f"{n} {label}")
    if not bits:
        return "no changes since last check"
    return "changes since last check: " + ", ".join(bits)


def changes_detail(changes, documents, limit=12):
    """The longer 'what actually changed' list, for the update panel."""
    if changes.get("first_run"):
        return "Baseline saved. From now on every launch reports what changed."
    by_id = {d["id"]: d["metadata"] for d in documents}
    lines = []
    for key, symbol in (("added", "+"), ("updated", "~"), ("removed", "-")):
        ids = changes[key]
        if not ids:
            continue
        lines.append(f"**{key.title()} ({len(ids)})**")
        for i in ids[:limit]:
            meta = by_id.get(i)
            title = f" - {meta['title']}" if meta else ""
            lines.append(f"- `{symbol}` {i}{title}")
        if len(ids) > limit:
            lines.append(f"- ...and {len(ids) - limit} more")
    return "\n".join(lines) if lines else "No changes since the last check."

---
## 5 · Index — a Chroma vector database plus BM25

**Chunking.** One course is one chunk. Course documents are already short and self-contained
(median 481 characters, max 1,388), so splitting them would break a course apart for no gain and
cost the source cards their ability to name a single code. Document-level chunking is the strategy;
the `to_document()` step above is where it happens.

**Vectorizing.** MiniLM turns each chunk into 384 numbers. The vectors go into **Chroma**, a
persistent vector database stored under `CHROMA_DIR`, with cosine distance to match the normalized
embeddings.

**Two retrievers**, because course questions come in two flavours:

* **Chroma / dense embeddings** handle the vague ones — "classes about teaching kids to read".
* **BM25** handles the literal ones — `IT 214`, "prerequisites", "student teaching".

Their rankings are fused with **reciprocal rank fusion**, so a course only needs to convince one of
them to make the shortlist. A course code typed in the question is pinned straight to the top, and a
subject code adds a counted fact so the model can answer "how many…" without hallucinating.

Each course is stored under its own id with its fingerprint in the metadata, so a re-sync upserts
only the courses that changed and deletes the ones that left the catalog.

In [ ]:
import numpy as np

WORD_RE = re.compile(r"[a-z0-9]+")
CODE_RE = re.compile(r"\b([a-z]{2,5})\s*-?\s*(\d{3}[a-z0-9]{0,3})\b", re.I)


def tokenize(text):
    return WORD_RE.findall(text.lower())


def codes_in(query):
    """'what is it 214 about' -> {'IT214'}"""
    return {f"{m.group(1)}{m.group(2)}".upper() for m in CODE_RE.finditer(query or "")}


def load_embedder():
    """MiniLM if we can get it, otherwise None and the index falls back to TF-IDF."""
    try:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer(EMBED_MODEL)
    except Exception as exc:
        print(f"[index] dense embeddings unavailable ({type(exc).__name__}: {exc})")
        print("[index] falling back to TF-IDF for the semantic half - the app still works.")
        return None


def fetch_chroma_from_drive():
    """Copy the saved Chroma store to the local working copy, once per runtime."""
    if CHROMA_WORK == CHROMA_DIR or CHROMA_WORK.exists() or not CHROMA_DIR.exists():
        return
    import shutil
    shutil.copytree(CHROMA_DIR, CHROMA_WORK)
    print(f"[index] vector DB copied from {CHROMA_DIR} to a local working copy")


def save_chroma_to_drive():
    """Copy the working Chroma store back to Drive after it changes."""
    if CHROMA_WORK == CHROMA_DIR or not CHROMA_WORK.exists():
        return
    import shutil
    staging = CHROMA_DIR.with_name(CHROMA_DIR.name + ".saving")
    shutil.rmtree(staging, ignore_errors=True)
    shutil.copytree(CHROMA_WORK, staging)          # copy first: a cut-off copy never replaces a good one
    shutil.rmtree(CHROMA_DIR, ignore_errors=True)
    staging.rename(CHROMA_DIR)
    print(f"[index] vector DB saved to {CHROMA_DIR}")


def open_collection():
    """The Chroma collection - created on first use, reopened from disk after that."""
    import chromadb
    fetch_chroma_from_drive()                      # a fresh runtime starts from the Drive copy
    client = chromadb.PersistentClient(path=str(CHROMA_WORK))
    try:
        # cosine distance matches the L2-normalized vectors MiniLM produces
        return client.get_or_create_collection(
            name=COLLECTION_NAME, embedding_function=None,
            metadata={"hnsw:space": "cosine"})
    except Exception:
        return client.get_or_create_collection(
            name=COLLECTION_NAME, embedding_function=None)


def chroma_metadata(doc):
    """Chroma metadata values must be str / int / float / bool, so lists are joined."""
    m = doc["metadata"]
    return {
        "code": m["code"],
        "title": m["title"],
        "subject": m["subject_code"],
        "credits": m["credits"],
        "career": m["career"],
        "college": m["college"],
        "department": "; ".join(m["departments"]),
        "attributes": "; ".join(m["attributes"]),
        "url": m["url"],
        "catalog": m["catalog"],
        "section": m.get("section", "courses"),   # what the section filter queries
        "fetched_at": m.get("fetched_at", ""),    # when a web page was last read
        "fingerprint": fingerprint(doc),      # what makes the re-sync incremental
    }


class CatalogIndex:
    """Chroma (dense) + BM25 (keyword) over the course corpus."""

    EMBED_BATCH  = 256     # courses per MiniLM forward pass
    UPSERT_BATCH = 1000    # courses per Chroma write

    def __init__(self, embedder=None):
        self.documents = []
        self.embedder = embedder
        self.collection = None        # the Chroma collection, once built
        self.position = {}            # doc id -> index in self.documents
        self.subjects = Counter()
        self._bm25 = None
        self._tfidf = None
        self._tfidf_matrix = None

    # --- building ------------------------------------------------------------

    def build(self, documents, progress=None):
        say = progress or (lambda msg: None)
        self.documents = documents
        self.position = {d["id"]: i for i, d in enumerate(documents)}
        self.by_id = {d["id"]: d for d in documents}
        self.subjects = Counter(d["metadata"]["subject_code"] for d in documents)

        say("building keyword index")
        from rank_bm25 import BM25Okapi
        self._bm25 = BM25Okapi([tokenize(d["text"]) for d in documents])

        if self.embedder is not None:
            self._sync_vector_db(documents, say)
        else:
            say("building TF-IDF index")
            from sklearn.feature_extraction.text import TfidfVectorizer
            self._tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
            self._tfidf_matrix = self._tfidf.fit_transform([d["text"] for d in documents])

        say(f"index ready - {len(documents)} courses")
        return self

    def _sync_vector_db(self, documents, say):
        """Bring Chroma in line with the corpus: upsert what changed, drop what left."""
        self.collection = open_collection()

        stored = self.collection.get(include=["metadatas"])
        have = {doc_id: (meta or {}).get("fingerprint")
                for doc_id, meta in zip(stored["ids"], stored["metadatas"] or [])}
        want = {d["id"]: fingerprint(d) for d in documents}

        gone = [doc_id for doc_id in have if doc_id not in want]
        todo = [d for d in documents if have.get(d["id"]) != want[d["id"]]]

        if gone:
            say(f"removing {len(gone)} withdrawn courses from the vector DB")
            for i in range(0, len(gone), self.UPSERT_BATCH):
                self.collection.delete(ids=gone[i:i + self.UPSERT_BATCH])

        if not todo:
            say(f"vector DB already current - {self.collection.count()} vectors")
            if gone or not CHROMA_DIR.exists():
                save_chroma_to_drive()
            return

        say(f"embedding {len(todo)} new or changed courses "
            f"({len(documents) - len(todo)} already in the vector DB)")
        vectors = self.embedder.encode(
            [d["text"] for d in todo],
            batch_size=self.EMBED_BATCH, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=False,
        ).astype("float32")

        for i in range(0, len(todo), self.UPSERT_BATCH):
            batch = todo[i:i + self.UPSERT_BATCH]
            self.collection.upsert(
                ids=[d["id"] for d in batch],
                embeddings=vectors[i:i + len(batch)].tolist(),
                documents=[d["text"] for d in batch],
                metadatas=[chroma_metadata(d) for d in batch],
            )
        say(f"vector DB holds {self.collection.count()} vectors")
        save_chroma_to_drive()

    # --- searching -----------------------------------------------------------

    def in_sections(self, idx, sections):
        """True when this document belongs to one of the requested sections."""
        return not sections or self.documents[idx]["metadata"].get("section") in sections

    def _bm25_ranking(self, query, n, sections=None):
        scores = np.asarray(self._bm25.get_scores(tokenize(query)))
        order = [int(i) for i in np.argsort(scores)[::-1]]
        return [i for i in order if self.in_sections(i, sections)][:n]

    def _dense_ranking(self, query, n, sections=None):
        if self.collection is not None:
            vector = self.embedder.encode([query], convert_to_numpy=True,
                                          normalize_embeddings=True).astype("float32")[0]
            # Chroma filters inside the database, so the section filter costs nothing
            where = {"section": {"$in": list(sections)}} if sections else None
            found = self.collection.query(
                query_embeddings=[vector.tolist()],
                n_results=max(1, min(n, self.collection.count())),
                where=where,
                include=["distances"],
            )
            return [self.position[i] for i in found["ids"][0] if i in self.position]

        from sklearn.metrics.pairwise import cosine_similarity
        scores = cosine_similarity(self._tfidf.transform([query]), self._tfidf_matrix).ravel()
        order = [int(i) for i in np.argsort(scores)[::-1]]
        return [i for i in order if self.in_sections(i, sections)][:n]

    def _ranked(self, query, k, pool, rrf_k, sections):
        """One fused ranking, optionally restricted to some sections."""
        bm_rank = self._bm25_ranking(query, pool, sections)
        dn_rank = self._dense_ranking(query, pool, sections)

        fused = {}
        for rank, idx in enumerate(bm_rank):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)
        for rank, idx in enumerate(dn_rank):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)

        # A course code typed in the question outranks everything else.
        wanted = codes_in(query)
        pinned = [i for i, d in enumerate(self.documents)
                  if d["metadata"]["code"].upper() in wanted
                  and self.in_sections(i, sections)] if wanted else []
        for i in pinned:
            fused[i] = fused.get(i, 0.0) + 1.0

        order = sorted(fused, key=fused.get, reverse=True)[:k]
        results = []
        for idx in order:
            why = []
            if idx in pinned:
                why.append("course code match")
            if idx in bm_rank[:10]:
                why.append("keyword")
            if idx in dn_rank[:10]:
                why.append("semantic")
            results.append({
                "doc": self.documents[idx],
                "score": round(float(fused[idx]), 5),
                "why": ", ".join(why) or "hybrid",
            })
        return results

    def search(self, query, k=TOP_K, pool=40, rrf_k=60, sections=None):
        """Hybrid search.

        sections=["billing"] searches only that section. sections=None lets the
        question decide: if it uses billing/housing/dining words, half the slots
        are reserved for that section so the 4,842 courses cannot crowd it out.
        """
        query = (query or "").strip()
        if not query:
            return []
        if sections:
            return self._ranked(query, k, pool, rrf_k, list(sections))

        routed = route_sections(query)
        if not routed:
            return self._ranked(query, k, pool, rrf_k, None)

        quota = max(2, k // 2)                       # reserved for the routed section
        primary = self._ranked(query, quota, pool, rrf_k, sorted(routed))
        for hit in primary:
            hit["why"] = f"{hit['why']}, routed to {hit['doc']['metadata']['section']}"

        merged, seen = [], set()
        for hit in primary + self._ranked(query, k, pool, rrf_k, None):
            if hit["doc"]["id"] not in seen:
                seen.add(hit["doc"]["id"])
                merged.append(hit)
        return merged[:k]

    def facts_for(self, query):
        """Counted facts the retriever cannot express, e.g. 'IT: 112 courses'."""
        found = []
        tokens = {t.upper() for t in re.findall(r"[A-Za-z]{2,5}", query or "")}
        for subject in tokens & set(self.subjects):
            found.append(f"Subject {subject}: {self.subjects[subject]} active courses in this catalog.")
        return found[:4]

---
## 6 · Answer — Gemini, grounded in the retrieved courses

The model never sees the catalog. It sees the handful of course documents the retriever picked,
plus a short instruction telling it to stay inside them and say so when the answer is not there.
That is the whole point of RAG: the facts come from the catalog, the wording comes from the model.

The model name is not hard-coded. `rank_models()` asks your key which models it can actually use and
ranks them newest-Flash-first, so this notebook keeps working as Google renames things.

Flash models get busy. A `503 UNAVAILABLE` ("high demand") is temporary and common at peak hours, so
`answer_stream()` retries with backoff, then drops to the next model on the list. Only if every model
in the chain is down does it fall back to showing the matching courses, and the technical reason is
printed to the notebook output rather than into the chat.

In [ ]:
SYSTEM_PROMPT = """You are the Illinois State University course catalog assistant.

You answer questions about ISU courses using ONLY the catalog excerpts in CONTEXT.

Rules:
- Ground every claim in CONTEXT. Never invent a course, code, credit hour, prerequisite or department.
- If CONTEXT does not answer the question, say so plainly and suggest a better search.
- Write course codes with a space: IT 214, MAT 120.
- Be concise: a short paragraph, or a tight bulleted list when comparing several courses.
- Mention credit hours and level (Undergraduate / Graduate) when they matter.
- Prerequisites live at the end of a course description - quote them accurately or say none are listed.
- You are talking to a student or advisor, so be direct and practical.

CONTEXT mixes two kinds of source: course catalog entries, and pages from ISU websites
(billing, housing, dining). Two extra rules apply to the website pages:
- Money, rates, deadlines and dates: quote the figure exactly, name the page it came from, and give
  the "Last checked" date shown in that excerpt. Never state an amount that is not in CONTEXT.
- You cannot see anyone's account. Never state a student's balance, charges, housing assignment or
  meal plan - those are behind a login. Explain the process and point to the right page instead.
- If two sources disagree, say so and prefer the one checked more recently.
- Every excerpt carries a "Source:" URL. When you use one, link it: write the page name as a
  markdown link, e.g. [Payment plans](https://studentaccounts.illinoisstate.edu/billing/).
- End any answer drawn from a website page with a "Source:" line listing the pages you used as
  markdown links. Course answers need no link line - the source cards already carry them."""

# Three kinds of failure that need three different responses. Quota and "busy"
# look almost identical in a traceback but mean opposite things: a busy model
# clears in seconds, an exhausted quota does not clear until its window resets.
QUOTA_WORDS = ("429", "resource_exhausted", "quota", "exceeded your current")
BUSY_WORDS  = ("503", "unavailable", "overloaded", "high demand", "500", "internal",
               "deadline", "timeout", "temporarily")


def error_kind(exc):
    """'quota', 'busy' or 'permanent' - decides whether retrying is worth anything."""
    blob = f"{type(exc).__name__} {exc}".lower()
    if any(word in blob for word in QUOTA_WORDS):
        return "quota"
    if any(word in blob for word in BUSY_WORDS):
        return "busy"
    return "permanent"


def is_transient(exc):
    return error_kind(exc) in ("quota", "busy")


def get_api_key():
    """Colab secret, then environment variable."""
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            return key.strip()
    except Exception:
        pass
    return (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY") or "").strip()


def rank_models(client, limit=3):
    """Models this key can see, best first. The extras are the fallback chain."""
    names = []
    try:
        for m in client.models.list():
            actions = getattr(m, "supported_actions", None) or \
                      getattr(m, "supported_generation_methods", None) or []
            if "generateContent" in actions:
                names.append(m.name.split("/")[-1])
    except Exception as exc:
        print(f"[gemini] could not list models ({exc}) - using gemini-flash-latest")
        return ["gemini-flash-latest"]

    def version(name):
        m = re.search(r"gemini-(\d+)(?:\.(\d+))?", name)
        return (int(m.group(1)), int(m.group(2) or 0)) if m else (0, 0)

    def rank(name):
        noisy = any(w in name for w in ("preview", "exp", "thinking", "image", "tts", "live"))
        if re.search(r"flash$", name):          tier = 0
        elif re.search(r"flash-lite$", name):   tier = 1
        elif re.search(r"flash", name):         tier = 2
        elif re.search(r"pro$", name):          tier = 3
        else:                                   tier = 4
        major, minor = version(name)
        return (int(noisy), tier, -major, -minor, len(name))

    names.sort(key=rank)

    # Take the best of each TIER rather than the top three overall. Three Flash
    # models draw on one capacity pool, so a busy Flash usually means all three
    # are busy; a Flash-Lite is a genuinely different fallback.
    chosen, tiers = [], set()
    for name in names:
        tier = rank(name)[1]
        if tier not in tiers:
            tiers.add(tier)
            chosen.append(name)
        if len(chosen) >= limit:
            break

    chosen = chosen or names[:limit] or ["gemini-flash-latest"]
    print(f"[gemini] using {chosen[0]}" +
          (f" (fallbacks: {', '.join(chosen[1:])})" if len(chosen) > 1 else ""))
    return chosen


def probe_models(client, names):
    """One tiny call per model, so a model this key cannot actually use is found
    at startup rather than in front of an audience. Costs a handful of tokens."""
    working = []
    for name in names:
        try:
            client.models.generate_content(model=name, contents="ping")
            working.append(name)
        except Exception as exc:
            print(f"[gemini] dropping {name} - {error_kind(exc)}: {str(exc)[:90]}")
    if not working:
        print("[gemini] every model failed its probe; keeping the list and trying anyway")
    return working or list(names)


class Answerer:
    """Turns a question + retrieved courses into a grounded answer."""

    RETRIES = 3          # attempts per model before moving to the next one
    BACKOFF = 1.5        # seconds, multiplied by the attempt number

    def __init__(self, index):
        self.index = index
        self.client = None
        self.models = []
        self.error = None

        key = get_api_key()
        if not key:
            self.error = ("No GOOGLE_API_KEY found. In Colab: key icon -> Add new secret -> "
                          "name GOOGLE_API_KEY -> paste your aistudio.google.com/apikey key "
                          "-> turn on Notebook access.")
            return
        try:
            from google import genai
            self.client = genai.Client(api_key=key)
            self.models = [GEMINI_MODEL] if GEMINI_MODEL else rank_models(self.client)
            if PROBE_MODELS and not GEMINI_MODEL:
                self.models = probe_models(self.client, self.models)
        except Exception as exc:
            self.client = None
            self.error = f"Gemini client failed to start: {type(exc).__name__}: {exc}"

    @property
    def available(self):
        return self.client is not None

    @property
    def model(self):
        return self.models[0] if self.models else None

    # --- prompt ---------------------------------------------------------------

    def build_prompt(self, question, hits, history=None):
        blocks = [f"[{i}] {hit['doc']['text']}" for i, hit in enumerate(hits, 1)]
        facts = self.index.facts_for(question)

        parts = []
        if history:
            recent = "\n".join(f"{turn['role']}: {turn['content']}" for turn in history[-4:])
            parts.append(f"RECENT CONVERSATION\n{recent}")
        if facts:
            parts.append("CATALOG FACTS\n" + "\n".join(facts))
        parts.append("CONTEXT - Illinois State University catalog excerpts\n" + "\n\n".join(blocks))
        parts.append(f"QUESTION\n{question}")
        return "\n\n".join(parts)

    # --- answering ------------------------------------------------------------

    def _stream_once(self, model, prompt):
        from google.genai import types
        config = types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.2,
            max_output_tokens=1200,
        )
        for chunk in self.client.models.generate_content_stream(
                model=model, contents=prompt, config=config):
            piece = getattr(chunk, "text", None)
            if piece:
                yield piece

    def answer_stream(self, question, history=None, k=TOP_K, sections=None):
        """Yields (partial answer, hits). Retries a busy model, then tries the next one."""
        hits = self.index.search(question, k=k, sections=sections)
        if not hits:
            yield ("Nothing in the catalog matched that. Try a course code (IT 214), a subject, "
                   "or a topic like machine learning."), []
            return

        if not self.available:
            yield matches_only(hits, need_key=True), hits
            return

        prompt = self.build_prompt(question, hits, history)

        last_kind = None
        for model in self.models:
            for attempt in range(1, self.RETRIES + 1):
                text = ""
                try:
                    for piece in self._stream_once(model, prompt):
                        text += piece
                        yield text, hits
                    if text.strip():
                        return                      # done
                    raise RuntimeError("the model returned an empty response")
                except Exception as exc:
                    # The reason goes to the notebook output, never into the chat.
                    last_kind = error_kind(exc)
                    print(f"[gemini] {model} attempt {attempt} ({last_kind}): "
                          f"{type(exc).__name__}: {exc}")
                    if text.strip():
                        return                      # keep the partial answer, say nothing
                    if last_kind == "permanent":
                        break                       # bad key or bad request - next model
                    if last_kind == "quota":
                        break                       # retrying a quota error only burns more
                    if attempt < self.RETRIES:
                        time.sleep(self.BACKOFF * (2 ** (attempt - 1)))   # 1.5s, 3s, 6s

        yield matches_only(hits, problem=last_kind or "busy"), hits

    def answer(self, question, history=None, k=TOP_K, sections=None):
        """Blocking version - returns (text, hits)."""
        last = ("", [])
        for last in self.answer_stream(question, history, k, sections):
            pass
        return last


def matches_only(hits, need_key=False, problem=None):
    """Shown when no answer can be written: the retrieval, plainly, with no error text."""
    if problem == "quota":
        lines = ["**Gemini quota is used up for now.** Here are the catalog matches - written "
                 "answers come back when the quota window resets.", ""]
    elif problem == "permanent":
        lines = ["**Gemini rejected the request.** Here are the catalog matches - check the "
                 "notebook output for the reason.", ""]
    elif problem:
        lines = ["**The model is busy right now.** Here are the catalog matches for your question:", ""]
    else:
        lines = ["**Closest matches in the catalog**", ""]

    for hit in hits[:SHOW_SOURCES]:
        meta = hit["doc"]["metadata"]
        credits = f" - {meta['credits']} cr" if meta["credits"] else ""
        label = meta["code"] or meta.get("section", "").title()
        lines.append(f"**{label}** - [{meta['title']}]({meta['url']}){credits}")
        lines.append((meta["description"] or "No description published.")[:240] + "...")
        lines.append("")

    if need_key:
        lines.append("<sub>Add a GOOGLE_API_KEY secret in Colab for written answers.</sub>")
    return "\n".join(lines)

---
## 7 · App state — collect, diff, index, answer

`CatalogApp.sync()` is the whole pipeline in one method, and it is what the launch cell and the
**Check catalog for updates** button both call. If the catalog API is unreachable it falls back to
the cached corpus instead of failing, so a flaky network downgrades the demo rather than ending it.

In [ ]:
class CatalogApp:
    """Everything the interface needs: documents, index, answerer, freshness."""

    def __init__(self):
        self.documents = []
        self.meta = {}
        self.changes = {"first_run": True, "added": [], "removed": [], "updated": []}
        self.sections = Counter()          # how many chunks each section contributed
        self.index = CatalogIndex(embedder=load_embedder())
        self.answerer = None
        self.offline = False

    def load_cached_documents(self):
        if not CORPUS_PATH.exists():
            return []
        with open(CORPUS_PATH, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]

    def sync(self, progress=None):
        """Fetch -> diff -> snapshot -> (re)index. Returns the change record."""
        say = progress or (lambda msg: None)
        restore_from_drive()             # a previous mirror, if this runtime is fresh
        previous = load_snapshot()

        try:
            documents, meta = collect_catalog(progress=say)
            if WEB_SOURCES:                          # v2: billing, housing, dining
                documents = documents + collect_website(progress=say)
            self.offline = False
        except Exception as exc:
            cached = self.load_cached_documents()
            if not cached:
                raise
            say(f"catalog unreachable ({type(exc).__name__}) - using the cached copy")
            documents, meta = cached, dict(previous.get("meta") or {})
            meta["checked_at"] = meta.get("checked_at", "unknown")
            self.offline = True

        self.changes = diff_documents(documents, previous)
        say(describe_changes(self.changes))

        self.documents = documents
        self.meta = meta
        self.sections = Counter(d["metadata"].get("section", "courses") for d in documents)
        if not self.offline:
            save_snapshot(documents, meta)

        self.index.build(documents, progress=say)
        # Re-check for a Gemini key on every sync, so adding the Colab secret and
        # pressing "Check catalog for updates" is enough to leave retrieval-only mode.
        if self.answerer is None or not self.answerer.available:
            self.answerer = Answerer(self.index)

        mirror_to_drive()                # keep Drive in step with what we just built
        return self.changes

    def load(self, progress=None):
        """The assistant's startup: use what the crawler saved. Makes no catalog or
        website requests. Run sync() - the crawler - when you want fresh data."""
        say = progress or (lambda msg: None)
        restore_from_drive()             # a previous mirror, if this runtime is fresh

        documents = self.load_cached_documents()
        if not documents:
            raise RuntimeError(
                f"No saved data in {CACHE_DIR.resolve()}. Run the crawler cell first - "
                "the assistant never crawls on its own.")

        previous = load_snapshot()
        self.documents = documents
        self.meta = dict(previous.get("meta") or {})
        self.sections = Counter(d["metadata"].get("section", "courses") for d in documents)
        self.changes = diff_documents(documents, previous)   # the saved copy: "no changes"
        self.offline = True                                   # status strip says "cached copy"
        say(f"loaded {len(documents)} saved documents from {CACHE_DIR} - no crawling")

        self.index.build(documents, progress=say)             # BM25 in memory; Chroma reopened
        if self.answerer is None or not self.answerer.available:
            self.answerer = Answerer(self.index)
        return self.changes

    # --- small helpers the UI reads -----------------------------------------

    @property
    def checked_at(self):
        stamp = self.meta.get("checked_at", "")
        try:
            return datetime.fromisoformat(stamp).strftime("%b %d, %Y at %-I:%M %p")
        except Exception:
            return stamp or "unknown"

    def section_summary(self):
        """'4,842 courses, 312 billing, 180 housing' for the status strip."""
        return ", ".join(f"{n:,} {name}"
                         for name, n in sorted(self.sections.items(), key=lambda kv: -kv[1]))

    def ask(self, question, k=TOP_K, sections=None):
        """Headless question answering - handy for testing without the UI."""
        text, hits = self.answerer.answer(question, k=k, sections=sections)
        print(text, "\n")
        print("sources:")
        for hit in hits[:SHOW_SOURCES]:
            m = hit["doc"]["metadata"]
            print(f"  {m['code']:<10} {m['title'][:60]:<62} {m['url']}")
        return text

---
## 8 · Interface — Illinois State red and white

Brand colors straight from [ISU's design system](https://designsystem.illinoisstate.edu/guidelines/color/):
red `#CC0000`, white `#FFFFFF`, black text, with the official secondary blue `#56758F` kept for links only.

In [ ]:
os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")   # no phoning home on launch
import gradio as gr

# (button label, question actually asked)
EXAMPLES = [
    ("IT 214 prerequisites",   "What does IT 214 cover, and what are its prerequisites?"),
    ("Machine learning",       "Which courses teach machine learning or data science?"),
    ("Paying my bill",         "How do I pay my student account bill, and when is it due?"),
    ("Housing options",        "What on-campus housing options are there for first-year students?"),
    ("Meal plans",             "How do meal plans work and where can I use them?"),
]

# The dropdown label -> which sections retrieval is allowed to look at.
SECTION_CHOICES = {
    "Everything": None,
    "Courses":    ["courses"],
    "Billing":    ["billing"],
    "Housing":    ["housing"],
    "Dining":     ["dining"],
}

ISU_CSS = """
:root { --isu-red:#CC0000; --isu-red-dark:#A80000; --isu-blue:#56758F; --isu-line:#E4E4E4; }

.gradio-container { max-width: 960px !important; margin: 0 auto !important; background:#FFFFFF !important; }
.gradio-container .html-container { padding-left:0 !important; padding-right:0 !important; }  /* header + status line up with the chat */
footer { display:none !important; }

/* Colab (and any browser set to dark) hands Gradio a dark theme, which paints text
   white - invisible on this app's white bubbles. The app is always light, so the
   colors are pinned here and the ISU palette is re-asserted at the end of the file. */
html, body, gradio-app, .gradio-container, .app, .main, .contain,
.gradio-container .block, .gradio-container .form, .gradio-container .wrap,
.gradio-container .panel, .bubble-wrap, .icon-button-wrapper,
.gradio-container textarea, .gradio-container input {
    background-color:#FFFFFF !important; }
.gradio-container button.secondary { background:#F3F3F3 !important; color:#111111 !important;
    border:1px solid #DDDDDD !important; }
.gradio-container, .gradio-container p, .gradio-container span, .gradio-container li,
.gradio-container td, .gradio-container th, .gradio-container h1, .gradio-container h2,
.gradio-container h3, .gradio-container h4, .gradio-container strong, .gradio-container em,
.gradio-container code, .gradio-container label, .gradio-container textarea,
.gradio-container input, .gradio-container .prose, .gradio-container .md {
    color:#111111 !important; }
.gradio-container button.primary, .gradio-container button.primary * { color:#FFFFFF !important; }

/* --- header band --- */
.isu-header { background:var(--isu-red); color:#fff; border-radius:12px 12px 0 0;
              padding:22px 26px 20px; }
.isu-header .eyebrow { font-size:11px; letter-spacing:.16em; text-transform:uppercase;
                       opacity:.9; font-weight:700; }
.isu-header h1 { margin:6px 0 4px; font-size:28px; font-weight:800; line-height:1.15; color:#fff; }
.isu-header p  { margin:0; font-size:14px; opacity:.92; }

/* --- status strip --- */
.isu-status { border:1px solid var(--isu-line); border-top:none; border-radius:0 0 12px 12px;
              background:#FAFAFA; padding:12px 18px; font-size:13px; color:#333; }
.isu-status b { color:#000; }
.isu-dot { display:inline-block; width:8px; height:8px; border-radius:50%;
           background:var(--isu-red); margin-right:8px; vertical-align:middle; }
.isu-status .sep { color:#BBB; margin:0 9px; }

/* --- chat --- */
.isu-chat { border:1px solid var(--isu-line) !important; border-radius:12px !important; }
.isu-chat .message.user, .isu-chat [data-testid="user"] {
    background:var(--isu-red) !important; border-color:var(--isu-red) !important; }
.isu-chat .message.user, .isu-chat .message.user *,
.isu-chat [data-testid="user"], .isu-chat [data-testid="user"] * { color:#fff !important; }
.isu-chat .message.bot, .isu-chat [data-testid="bot"] {
    background:#fff !important; color:#111 !important; border:1px solid var(--isu-line) !important; }
/* Gradio nests a second bordered box inside each bubble - flatten it so a message
   is one clean card instead of a box inside a box. */
.isu-chat .message > div, .isu-chat .message .latest, .isu-chat .message-content {
    border:none !important; background:transparent !important; box-shadow:none !important; }
.isu-chat .message.bot, .isu-chat .message.user {
    padding:10px 14px !important; border-radius:10px !important; }
.isu-chat a { color:var(--isu-blue); }

/* --- starter chips --- */
.isu-chips { flex-wrap:wrap !important; gap:8px !important; }
.isu-chip { background:#fff !important; color:var(--isu-red) !important;
            border:1px solid var(--isu-red) !important; border-radius:18px !important;
            font-size:12px !important; font-weight:600 !important;
            padding:7px 15px !important; min-width:0 !important; width:auto !important;
            height:auto !important; flex:0 0 auto !important; white-space:nowrap !important; }
.isu-chip:hover { background:var(--isu-red) !important; color:#fff !important; }
.isu-tools { gap:8px !important; }
.isu-tools button { flex:0 0 auto !important; width:auto !important; min-width:0 !important;
                    font-size:12px !important; }

/* --- source cards --- */
.isu-panel-title { font-size:12px; letter-spacing:.12em; text-transform:uppercase;
                   font-weight:800; color:var(--isu-red); margin:2px 0 10px; }
.isu-card { border:1px solid var(--isu-line); border-left:4px solid var(--isu-red);
            border-radius:10px; padding:12px 14px; margin-bottom:10px; background:#fff; }
.isu-card .code { font-weight:800; color:var(--isu-red); font-size:14px; letter-spacing:.02em; }
.isu-card .tag  { float:right; font-size:11px; color:#555; background:#F3F3F3;
                  border-radius:999px; padding:2px 9px; margin-left:6px; }
.isu-card .title { font-weight:700; font-size:14px; margin:4px 0 2px; color:#111; }
.isu-card .meta  { font-size:12px; color:#666; margin-bottom:6px; }
.isu-card .desc  { font-size:12.5px; color:#333; line-height:1.45; margin:0 0 8px; }
.isu-card a { font-size:12px; font-weight:700; color:var(--isu-blue); text-decoration:none; }
.isu-card a:hover { color:var(--isu-red); }
.isu-empty { border:1px dashed var(--isu-line); border-radius:10px; padding:18px;
             color:#888; font-size:13px; text-align:center; }

/* --- the ISU palette, asserted last so it beats the forced-light block --- */
.isu-header, .isu-header * { color:#FFFFFF !important; }
.isu-status, .isu-status * { color:#333333 !important; }
.isu-status b { color:#000000 !important; }
.isu-status .sep { color:#BBBBBB !important; }
.gradio-container button.isu-chip, .isu-chip, .isu-chip * {
    background:#FFFFFF !important; color:var(--isu-red) !important;
    border-color:var(--isu-red) !important; }
.gradio-container button.isu-chip:hover, .isu-chip:hover, .isu-chip:hover * {
    background:var(--isu-red) !important; color:#FFFFFF !important; }
.isu-panel-title { color:var(--isu-red) !important; }
.isu-card .code { color:var(--isu-red) !important; }
.isu-card .title { color:#111111 !important; }
.isu-card .meta, .isu-card .tag { color:#555555 !important; }
.isu-card .desc { color:#333333 !important; }
.isu-card a { color:var(--isu-blue) !important; }
.isu-card a:hover { color:var(--isu-red) !important; }
.isu-empty { color:#888888 !important; }
.isu-chat .message.bot, .isu-chat .message.bot * { color:#111111 !important; }
.isu-chat .message.user, .isu-chat .message.user * { color:#FFFFFF !important; }
"""

ISU_RED_HUE = gr.themes.Color(
    c50="#FFF1F1", c100="#FFE0E0", c200="#FFC4C4", c300="#FF9B9B", c400="#F5605F",
    c500="#E01F1F", c600="#CC0000", c700="#A80000", c800="#8A0000", c900="#6E0000",
    c950="#3D0000",
)

ISU_THEME = gr.themes.Base(
    primary_hue=ISU_RED_HUE,
    neutral_hue=gr.themes.colors.gray,
    font=["Inter", "system-ui", "-apple-system", "Segoe UI", "sans-serif"],
).set(
    body_background_fill="#FFFFFF",
    block_background_fill="#FFFFFF",
    border_color_primary="#E4E4E4",
    button_primary_background_fill="#CC0000",
    button_primary_background_fill_hover="#A80000",
    button_primary_text_color="#FFFFFF",
)


# Gradio 6 moved theme/css from Blocks() to launch() and made "messages" the only
# chat format. Both generations are supported so this notebook does not rot.
GRADIO_MAJOR  = int(gr.__version__.split(".")[0])
BLOCKS_STYLE  = {} if GRADIO_MAJOR >= 6 else {"theme": ISU_THEME, "css": ISU_CSS}
LAUNCH_STYLE  = {"theme": ISU_THEME, "css": ISU_CSS} if GRADIO_MAJOR >= 6 else {}
CHAT_KWARGS   = {} if GRADIO_MAJOR >= 6 else {"type": "messages"}


def esc(text):
    return html_lib.escape(str(text or ""))


def header_html():
    return """
    <div class="isu-header">
      <div class="eyebrow">Illinois State University</div>
      <h1>ReggieBot</h1>
      <p>Ask your favorite red bird questions about ISU courses, billing, meal plans, and housing.</p>
    </div>"""


def status_html(app):
    meta = app.meta
    mode = "catalog matches only" if (app.answerer and not app.answerer.available) else "RAG"
    state = "cached copy" if app.offline else "live"
    return (
        '<div class="isu-status"><span class="isu-dot"></span>'
        f'<b>{esc(meta.get("catalog_name", "Catalog"))}</b>'
        f'<span class="sep">|</span>{len(app.documents):,} chunks'
        f'<span class="sep">|</span>{esc(app.section_summary())}'
        f'<span class="sep">|</span>checked {esc(app.checked_at)} ({state})'
        f'<span class="sep">|</span>{esc(describe_changes(app.changes))}'
        f'<span class="sep">|</span>mode: {mode}</div>'
    )


EMPTY_SOURCES = ('<div class="isu-empty">Ask a question and the courses behind the answer '
                 'will show up here, with links into the catalog.</div>')


def source_cards_html(hits):
    if not hits:
        return EMPTY_SOURCES
    cards = []
    for hit in hits[:SHOW_SOURCES]:
        m = hit["doc"]["metadata"]
        desc = m["description"] or "No catalog description published for this course."
        if len(desc) > 220:
            desc = desc[:220].rsplit(" ", 1)[0] + "..."
        tags = ""
        if m.get("fetched_at"):                    # web pages: show when we last read them
            tags += f'<span class="tag">checked {esc(m["fetched_at"][:10])}</span>'
        if m["credits"]:
            tags += f'<span class="tag">{esc(m["credits"])} cr</span>'
        if m["career"]:
            tags += f'<span class="tag">{esc(m["career"])}</span>'
        dept = (m["departments"] or [m["college"]] or [""])[0] or m.get("host", "")
        cards.append(f"""
        <div class="isu-card">
          <div><span class="code">{esc(m['code'] or m.get('section', '').title())}</span>{tags}</div>
          <div class="title">{esc(m['title'])}</div>
          <div class="meta">{esc(dept)}</div>
          <p class="desc">{esc(desc)}</p>
          <a href="{esc(m['url'])}" target="_blank" rel="noopener">View in the ISU catalog &rarr;</a>
        </div>""")
    return "\n".join(cards)


# --- source links under each answer ----------------------------------------------
SOURCES_RULE = "\n\n---\n"          # the line between an answer and its source list
ALREADY_LISTED = ("**The model is busy", "**Gemini quota", "**Gemini rejected",
                  "**Closest matches", "Nothing in the catalog matched")   # these already show their links
LINK_RE = re.compile(r"https?://[^\s)\]>\"']+")
LINK_ONLY_LINE = re.compile(r"^\s*(?:[-*]|\d+\.)?\s*\[[^\]]+\]\(https?://[^)]+\)\s*[,.;]?\s*$")
SOURCE_LINE = re.compile(r"^\s*\**\s*sources?\s*\**\s*:", re.I)


def source_label(meta):
    """'IT 214 - Social, Legal, ...' for a course; 'FAQ - Refunds' for a web page."""
    if meta.get("code"):
        label = re.sub(r"^([A-Z]+)(\d)", r"\1 \2", meta["code"]) + " - " + meta["title"]
    else:
        label = " - ".join(p for p in (meta.get("title"), meta.get("heading")) if p) or meta["url"]
    return label.replace("[", "(").replace("]", ")")


def answer_sources(text, hits):
    """The retrieved pages an answer drew on: courses it names and pages it links, once each."""
    named = re.sub(r"[\s\-]", "", text).upper()
    linked = {url.rstrip("/.,") for url in LINK_RE.findall(text)}
    used, seen = [], set()
    for hit in hits:
        meta = hit["doc"]["metadata"]
        url = meta["url"].rstrip("/")
        cited = (meta.get("code") and meta["code"].upper() in named) or url in linked
        if cited and url not in seen:
            seen.add(url)
            used.append(meta)
    return used


def strip_source_line(text):
    """Drop a closing 'Source: ...' line the model wrote; the list underneath replaces it."""
    lines = text.rstrip().split("\n")
    end = len(lines)
    while end > 0 and LINK_ONLY_LINE.match(lines[end - 1]):
        end -= 1
    if end > 0 and SOURCE_LINE.match(lines[end - 1]):
        return "\n".join(lines[:end - 1]).rstrip()
    return text.rstrip()


def with_sources(text, hits):
    """The finished answer with its source links listed directly underneath."""
    if not hits or text.startswith(ALREADY_LISTED):
        return text
    used = answer_sources(text, hits)
    heading = "**Sources**"
    if not used:                       # the answer named nothing: show where to look instead
        heading, seen = "**Closest matches in the catalog**", set()
        for hit in hits:
            meta = hit["doc"]["metadata"]
            if meta["url"].rstrip("/") not in seen and len(used) < 3:
                seen.add(meta["url"].rstrip("/"))
                used.append(meta)
    links = "\n".join(f"{i}. [{source_label(m)}]({m['url']})" for i, m in enumerate(used[:SHOW_SOURCES], 1))
    return strip_source_line(text) + SOURCES_RULE + heading + "\n\n" + links


def without_sources(content):
    """An earlier answer minus its source list, before it goes back to Gemini as conversation history."""
    if isinstance(content, str):
        return content.rsplit(SOURCES_RULE, 1)[0]
    if isinstance(content, list):      # Gradio 6 returns messages as a list of content blocks
        return [{**block, "text": without_sources(block["text"])}
                if isinstance(block, dict) and isinstance(block.get("text"), str) else block
                for block in content]
    return content


def build_ui(app):
    with gr.Blocks(title="ReggieBot", **BLOCKS_STYLE) as demo:
        gr.HTML(header_html())
        status = gr.HTML(status_html(app))

        with gr.Row():
            with gr.Column(scale=3):
                chat = gr.Chatbot(
                    height=430, show_label=False, elem_classes="isu-chat", **CHAT_KWARGS,
                    value=[{"role": "assistant",
                            "content": "Ask me about any Illinois State course - by code, by topic, "
                                       "or by what you still need to graduate."}],
                )
                with gr.Row():
                    box = gr.Textbox(placeholder="e.g. When is my tuition bill due?",
                                     show_label=False, scale=6, autofocus=True, container=False)
                    send = gr.Button("Ask", variant="primary", scale=1, min_width=90)
                with gr.Row(elem_classes="isu-chips"):
                    chips = [gr.Button(label, elem_classes="isu-chip", size="sm")
                             for label, _ in EXAMPLES]
                with gr.Row(elem_classes="isu-tools"):
                    clear = gr.Button("Clear conversation", size="sm", variant="secondary")
                    refresh = gr.Button("Check catalog for updates", size="sm", variant="secondary")

        with gr.Accordion("What changed in the catalog", open=False):
            change_md = gr.Markdown(changes_detail(app.changes, app.documents))

        # --- behaviour --------------------------------------------------------

        def respond(message, history):
            message = (message or "").strip()
            if not message:
                yield history, ""
                return
            history = list(history) + [
                {"role": "user", "content": message},
                {"role": "assistant", "content": "_Searching the catalog..._"},
            ]
            yield history, ""

            # earlier answers go back to Gemini without the source list added under them
            prior = [{**turn, "content": without_sources(turn.get("content"))} if isinstance(turn, dict) else turn
                     for turn in history[:-2]]
            text, hits = "", []
            for text, hits in app.answerer.answer_stream(message, prior):   # the router picks the section
                history[-1] = {"role": "assistant", "content": text}
                yield history, ""
            history[-1] = {"role": "assistant", "content": with_sources(text, hits)}   # links under the answer
            yield history, ""

        def do_refresh():
            yield gr.update(value='<div class="isu-status"><span class="isu-dot"></span>'
                                  'Checking the catalog for changes...</div>'), gr.update()
            app.sync()
            yield status_html(app), changes_detail(app.changes, app.documents)

        io = ([box, chat], [chat, box])
        send.click(respond, *io)
        box.submit(respond, *io)
        for chip, (_, question) in zip(chips, EXAMPLES):
            chip.click(lambda q=question: q, None, box).then(respond, *io)
        clear.click(lambda: ([], ""), None, [chat, box])
        refresh.click(do_refresh, None, [status, change_md])

    return demo

---
## 9 · Crawler — run this when you want fresh data

This is the only cell that talks to the catalog API or crawls a website. It fetches everything,
reports what changed since last time, re-embeds only what moved, and saves the result to
`CACHE_DIR` (`corpus.jsonl`, `snapshot.json`, and the Chroma store under `chroma/`).

Tick **REFRESH_DATA** and run the cell to crawl. Leave it unticked and **Run all** skips this cell,
so the assistant starts from the saved data without touching the network.

In [ ]:
REFRESH_DATA = False  #@param {type:"boolean"}

if REFRESH_DATA:
    CRAWLER = CatalogApp()
    CRAWLER.sync(progress=lambda msg: print("  ", msg))
    print("\n" + describe_changes(CRAWLER.changes))
    print(f"saved to {CACHE_DIR.resolve()}")
else:
    print("crawler skipped - tick REFRESH_DATA above and run this cell to fetch fresh data")

---
## 10 · Assistant — run this normally

Loads what the crawler saved and opens the app. No catalog or website requests are made here;
if nothing has been saved yet, it stops and tells you to run the crawler. Gradio prints two links —
use the `gradio.live` one to show the demo on another device.

The **Check catalog for updates** button inside the app still runs the crawler on demand, so a
refresh is always one deliberate click away.

In [ ]:
APP = CatalogApp()
APP.load(progress=lambda msg: print("  ", msg))

print("\n" + describe_changes(APP.changes))
if APP.answerer and not APP.answerer.available:
    print("!", APP.answerer.error)

demo = build_ui(APP)
demo.queue().launch(share=SHARE_LINK, debug=False, **LAUNCH_STYLE)

---
## 11 · Ask without the interface

Useful for testing retrieval, or for screenshots of the pipeline working end to end.

In [ ]:
APP.ask("What are the prerequisites for IT 214?")

In [ ]:
# Retrieval on its own - what the model actually gets handed.
for hit in APP.index.search("cybersecurity and network defense", k=5):
    m = hit["doc"]["metadata"]
    print(f"{hit['score']:.4f}  {m['code']:<9} {m['title'][:58]:<60} [{hit['why']}]")

---
### Troubleshooting Gemini

"The model is busy" and "you are out of quota" arrive as different HTTP codes but look the same in a
traceback, and they need opposite responses:

| What you see in the notebook output | Code | What it means | What to do |
|---|---|---|---|
| `busy` | 503 | Google's capacity, not your setup | Wait a minute, or pin a Lite model |
| `quota` | 429 | The free tier window is used up | Wait for the reset, or use another key |
| `permanent` | 400 / 403 | Bad key, or this key cannot use that model | Pin a model that works |

`gemini_report()` tells you which one you have. It sends a one-word prompt to each model in the
fallback chain and prints the result, so you are reading facts instead of guessing.

In [ ]:
def gemini_report():
    """Tell 'busy' from 'out of quota' from 'bad key'. Costs a few tokens."""
    key = get_api_key()
    print("API key:", f"found, ending {key[-4:]}" if key else "NOT FOUND")
    if not key:
        print("Colab: key icon -> Add new secret -> GOOGLE_API_KEY -> Notebook access on.")
        return

    from google import genai
    client = genai.Client(api_key=key)

    try:
        visible = [m.name.split("/")[-1] for m in client.models.list()
                   if "generateContent" in (getattr(m, "supported_actions", None)
                                            or getattr(m, "supported_generation_methods", None)
                                            or [])]
        print(f"models this key can see: {len(visible)}")
    except Exception as exc:
        print(f"could not even list models ({error_kind(exc)}): {str(exc)[:140]}")
        print("that usually means the key itself is wrong or has no API access enabled.")
        return

    candidates = (APP.answerer.models if APP.answerer and APP.answerer.models
                  else rank_models(client))
    print(f"\ntrying {len(candidates)} model(s) with a one-word prompt:\n")
    for name in candidates:
        started = time.time()
        try:
            client.models.generate_content(model=name, contents="ping")
            print(f"  {name:<30} OK         {time.time() - started:.1f}s")
        except Exception as exc:
            print(f"  {name:<30} {error_kind(exc).upper():<10} {str(exc)[:110]}")

    print("""
  OK         the model works - the busy message was a passing spike, just ask again
  BUSY       503 from Google. Nothing is wrong with your setup. Retry, or pin a Lite model
  QUOTA      429. The free tier is used up - check aistudio.google.com/rate-limit
  PERMANENT  bad key, or this key cannot use that model

To pin a specific model, set it in the config cell and re-run from there:
  GEMINI_MODEL = "gemini-flash-lite-latest"      # Lite tiers are far less contended
""")


gemini_report()

---
## 12 · Publish the corpus and the vector database

Every sync writes three things into the cache folder:

| File | What it holds |
|---|---|
| `corpus.jsonl` | One JSON document per active course - `id`, `text`, `metadata` |
| `snapshot.json` | Per-course fingerprints, used for change detection |
| `chroma/` | The Chroma vector database - `chroma.sqlite3` plus its index segments |

Colab wipes its disk when the runtime disconnects, so the prototype needs a permanent home for
these. This cell pushes the whole folder to a **Hugging Face dataset repo**.

1. Set `HF_DATASET_REPO = "<your-username>/isu-course-catalog-rag"` in the config cell.
2. Create a **write** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. Add it as a Colab secret named `HF_TOKEN`, with notebook access on.

`restore_from_hub()` is the other direction: on a fresh runtime it pulls the corpus and vector DB
back down, so the app starts without re-fetching or re-embedding anything.

In [ ]:
DATASET_CARD = """---
license: other
language:
- en
tags:
- rag
- retrieval
- education
- course-catalog
---

# ISU Course Catalog - RAG corpus and vector store

Retrieval corpus and [Chroma](https://www.trychroma.com/) vector database behind an Illinois State
University course catalog assistant.

| File | What it is |
|---|---|
| `corpus.jsonl` | One JSON document per active course: `id`, `text`, `metadata` |
| `snapshot.json` | Per-course fingerprints used for change detection |
| `chroma/` | Persistent Chroma collection `{collection}`, cosine space |

| | |
|---|---|
| Courses | {count} |
| Subjects | {subjects} |
| Catalog | {catalog} |
| Built | {built} |
| Embedding model | `{model}` |
| Chunking | one course per chunk, no splitting |

Source: the public Coursedog API behind <https://catalog.illinoisstate.edu>. Rebuilt by the
notebook in this project; course text belongs to Illinois State University and this is not an
official ISU publication.
"""


def hf_token():
    """Colab secret first, then environment variable."""
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token.strip()
    except Exception:
        pass
    return (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip()


def write_dataset_card(app):
    """A README.md so the dataset repo explains itself on the Hub."""
    card = DATASET_CARD.format(
        collection=COLLECTION_NAME,
        count=f"{len(app.documents):,}",
        subjects=app.meta.get("subject_count", "-"),
        catalog=app.meta.get("catalog_name", "-"),
        built=app.meta.get("checked_at", "-"),
        model=EMBED_MODEL,
    )
    (CACHE_DIR / "README.md").write_text(card, encoding="utf-8")


def publish_to_hub(repo_id=None, private=None):
    """Push corpus.jsonl, snapshot.json and the Chroma store to a HF dataset repo."""
    repo_id = repo_id or HF_DATASET_REPO
    if not repo_id:
        print("Set HF_DATASET_REPO in the config cell to publish. Skipping.")
        return None

    token = hf_token()
    if not token:
        print("No HF_TOKEN found. Add it as a Colab secret (key icon) and re-run. Skipping.")
        return None

    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True,
                    private=HF_PRIVATE if private is None else private)

    write_dataset_card(APP)
    api.upload_folder(
        folder_path=str(CACHE_DIR),
        repo_id=repo_id,
        repo_type="dataset",
        commit_message=f"ISU catalog corpus + Chroma store ({len(APP.documents)} courses)",
    )
    url = f"https://huggingface.co/datasets/{repo_id}"
    print(f"published {len(APP.documents)} courses to {url}")
    return url


def restore_from_hub(repo_id=None):
    """Pull a previously published corpus + vector DB into the cache folder."""
    repo_id = repo_id or HF_DATASET_REPO
    if not repo_id:
        print("Set HF_DATASET_REPO in the config cell first.")
        return None

    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=repo_id, repo_type="dataset",
                      local_dir=str(CACHE_DIR), token=hf_token() or None)
    print(f"restored into {CACHE_DIR.resolve()} - re-run the launch cell to use it")
    return CACHE_DIR


publish_to_hub()

### Local copies

The corpus is a plain JSONL file in the cache folder, ready for any other RAG stack. Run this to
download it.

In [ ]:
print(CORPUS_PATH.resolve(), f"({len(APP.documents)} documents)")
print(CHROMA_DIR.resolve(), "(Chroma vector database)")

try:
    from google.colab import files
    files.download(str(CORPUS_PATH))
except ImportError:
    print("Not running in Colab - the files are in this notebook's folder.")

---
## How it fits together

```
catalog.illinoisstate.edu  ->  Coursedog JSON API  ->  4,800+ course records
                                                             |
                                    to_record / to_document  (1 course = 1 chunk)
                                                             v
                          fingerprint + snapshot.json  ->  added / updated / removed
                                                             |
                            BM25 index        +        Chroma vector DB (MiniLM, 384-dim)
                                                       upserted only where the fingerprint moved
                                                             v
                                       reciprocal rank fusion -> top 8 courses
                                                             |
                                    Gemini, told to use only those 8  ->  answer
                                                             v
                                  ReggieBot chat, source links under each answer
                                                             |
                                    corpus.jsonl + chroma/  ->  Hugging Face dataset repo
```

### Things worth tweaking

| Want | Change |
|---|---|
| Keep the cache between Colab sessions | `DRIVE_MODE = "drive"` (the default), or publish to the Hub and `restore_from_hub()` |
| Add another part of the site | one more entry in `WEB_SOURCES`, plus a label in `SECTION_CHOICES` |
| Crawl more of each site | raise `CRAWL_MAX_PAGES` and `CRAWL_MAX_DEPTH` |
| Swap the vector DB | `open_collection()` is the only place Chroma is touched |
| More context per answer | raise `TOP_K` (8 -> 12) |
| Pin a specific catalog year | set `CATALOG_ID` to one of the ids printed by `list_catalogs()` |
| A different model | set `GEMINI_MODEL = "gemini-3.5-flash"` (or any id your key can see) |
| Better semantic recall | swap `EMBED_MODEL` for `BAAI/bge-base-en-v1.5` (bigger, slower, sharper) |
| No public link | `SHARE_LINK = False` |

### Honest limits

* The catalog is the only source. Section times, seat availability and instructors live in the
  registration system, not here, so the assistant will say it does not know.
* Prerequisites are free text at the end of a description. The model quotes them; it does not
  reason over a prerequisite graph.
* "How many" questions are answered from counted catalog facts for a subject code; anything more
  complex should be checked against the corpus directly.

---
## 13 · Experiment — Arm 1: the assistant as built

Arm 1 is the **baseline**: the assistant exactly as the cells above build it — BM25 + Chroma retrieval
fused by rank, the course-code pin, the section router, the top 8 chunks, and Gemini. Nothing is changed
for the experiment, so each later arm can change one thing and be compared against these numbers.

**Held fixed across arms:** the saved data (do not re-crawl between arms), the question set below,
`TOP_K = 8`, the embedding model, and the Gemini model chain chosen at startup.

| Metric | What it means |
|---|---|
| **Hit@5** | The right source is among the 5 source cards the user sees |
| **Hit@8** | The right source is among the 8 chunks handed to Gemini |
| **MRR** | Average of 1 ÷ the rank of the first right source (1.0 = always first) |
| **Routing** | Billing, housing and dining questions are sent to their own section |
| **Answered** | Gemini wrote an answer instead of the busy / quota fallback |
| **Cites source** | The answer names the expected course code or links the expected website |
| **Declined** | For trap questions with no answer in the data, the reply says so instead of inventing one |

"The right source" is a rule attached to each question: a course code, or a section plus key phrases.
Every rule is checked against the loaded data before scoring, and a rule that matches nothing is reported
and left out rather than counted as a miss.

**To run:** start the assistant (§10), tick **`RUN_ARM1`** below and run the cells in this section.
Scoring retrieval takes seconds; the Gemini answers take roughly 6–10 minutes for 50 questions because the
cell pauses between calls to stay inside the free-tier rate limit. Everything is saved to
`experiments/arm1/` inside the cache folder (your Google Drive by default), and a write-up draft filled with
your numbers is shown at the end.

In [ ]:
# The Arm 1 question set. Each question carries the rule that decides which retrieved chunks
# count as "the right source":
#   codes    - a course whose code is in the list                      (course-code questions)
#   section + phrases - a chunk from that section containing any phrase  (topic and website questions)
#   trap     - nothing in the data answers it; the reply should say so rather than invent an answer
# Edit freely before the first run, then keep the list fixed so every arm answers the same questions.

ARM1_QUESTIONS = [
    # --- course codes: the code pin should put these first, whatever the spelling ------------------
    {"id": "C01", "category": "course code", "question": "What are the prerequisites for IT 214?", "codes": ["IT214"]},
    {"id": "C02", "category": "course code", "question": "What is IT 140 about?", "codes": ["IT140"]},
    {"id": "C03", "category": "course code", "question": "How many credit hours is IT 150?", "codes": ["IT150"]},
    {"id": "C04", "category": "course code", "question": "What's the difference between IT 354 and IT 454?", "codes": ["IT354", "IT454"]},
    {"id": "C05", "category": "course code", "question": "Describe TEC 283.", "codes": ["TEC283"]},
    {"id": "C06", "category": "course code", "question": "What does CTK303 cover?", "codes": ["CTK303"]},
    {"id": "C07", "category": "course code", "question": "what is com 110", "codes": ["COM110"]},
    {"id": "C08", "category": "course code", "question": "Tell me about ENG 101.", "codes": ["ENG101"]},
    {"id": "C09", "category": "course code", "question": "What is IT-344 about?", "codes": ["IT344"]},
    {"id": "C10", "category": "course code", "question": "Is MAT 120 an undergraduate or graduate course?", "codes": ["MAT120"]},

    # --- course topics: no code to pin, so meaning has to carry it ---------------------------------
    {"id": "T01", "category": "course topic", "question": "Which courses teach web application development?", "section": "courses", "phrases": ["web application"]},
    {"id": "T02", "category": "course topic", "question": "Are there classes about the ethics of technology?", "section": "courses", "phrases": ["ethical", "ethics"]},
    {"id": "T03", "category": "course topic", "question": "What courses cover machine learning or artificial intelligence?", "section": "courses", "phrases": ["machine learning", "artificial intelligence"]},
    {"id": "T04", "category": "course topic", "question": "I want a class about protecting networks from hackers.", "section": "courses", "phrases": ["security", "cybersecurity"]},
    {"id": "T05", "category": "course topic", "question": "Which courses teach database design?", "section": "courses", "phrases": ["database"]},
    {"id": "T06", "category": "course topic", "question": "Are there courses on digital imaging?", "section": "courses", "phrases": ["imaging"]},
    {"id": "T07", "category": "course topic", "question": "Is there a class that teaches spreadsheets and word processing?", "section": "courses", "phrases": ["spreadsheet", "word processing"]},
    {"id": "T08", "category": "course topic", "question": "What statistics courses can I take?", "section": "courses", "phrases": ["statistic"]},
    {"id": "T09", "category": "course topic", "question": "Classes about teaching children to read", "section": "courses", "phrases": ["literacy", "reading instruction", "teaching reading", "teaching of reading"]},
    {"id": "T10", "category": "course topic", "question": "Are there courses on public speaking?", "section": "courses", "phrases": ["public speaking", "oral communication"]},

    # --- billing (studentaccounts.illinoisstate.edu) -----------------------------------------------
    {"id": "B01", "category": "billing", "question": "How do I sign up for a payment plan?", "section": "billing", "phrases": ["payment plan"]},
    {"id": "B02", "category": "billing", "question": "When will I get my refund?", "section": "billing", "phrases": ["refund"]},
    {"id": "B03", "category": "billing", "question": "How do I set up direct deposit for my refund?", "section": "billing", "phrases": ["erefund", "direct deposit"]},
    {"id": "B04", "category": "billing", "question": "Can my parents see my bill and pay it for me?", "section": "billing", "phrases": ["authorized user"]},
    {"id": "B05", "category": "billing", "question": "Can I pay tuition with a credit card?", "section": "billing", "phrases": ["credit card"]},
    {"id": "B06", "category": "billing", "question": "Where do I find my 1098-T tax form?", "section": "billing", "phrases": ["1098-t", "1098t"]},
    {"id": "B07", "category": "billing", "question": "What happens if my tuition payment is late?", "section": "billing", "phrases": ["late charge", "late fee", "delinquent"]},
    {"id": "B08", "category": "billing", "question": "Why was I charged for student health insurance?", "section": "billing", "phrases": ["health insurance"]},

    # --- housing (housing.illinoisstate.edu) -------------------------------------------------------
    {"id": "H01", "category": "housing", "question": "Do first-year students have to live on campus?", "section": "housing", "phrases": ["first two years", "live-on", "live on campus", "required to live"]},
    {"id": "H02", "category": "housing", "question": "What happens if I cancel my housing contract?", "section": "housing", "phrases": ["cancel"]},
    {"id": "H03", "category": "housing", "question": "What residence halls are there at ISU?", "section": "housing", "phrases": ["watterson", "hewett", "manchester", "tri-towers", "tri towers", "wilkins", "haynie"]},
    {"id": "H04", "category": "housing", "question": "Tell me about Watterson Towers.", "section": "housing", "phrases": ["watterson"]},
    {"id": "H05", "category": "housing", "question": "Who lives in Cardinal Court?", "section": "housing", "phrases": ["cardinal court"]},
    {"id": "H06", "category": "housing", "question": "When is the priority deadline for the housing application?", "section": "housing", "phrases": ["priority"]},
    {"id": "H07", "category": "housing", "question": "Is laundry free in the residence halls?", "section": "housing", "phrases": ["laundry", "washer"]},
    {"id": "H08", "category": "housing", "question": "Can I live on campus over the summer?", "section": "housing", "phrases": ["summer housing", "summer"]},

    # --- dining (dining.illinoisstate.edu) ---------------------------------------------------------
    {"id": "D01", "category": "dining", "question": "What meal plans are available?", "section": "dining", "phrases": ["meal plan"]},
    {"id": "D02", "category": "dining", "question": "Do I have to buy a meal plan if I live in a residence hall?", "section": "dining", "phrases": ["required to purchase", "required"]},
    {"id": "D03", "category": "dining", "question": "What are Flex Dollars?", "section": "dining", "phrases": ["flex dollars", "flex"]},
    {"id": "D04", "category": "dining", "question": "Where can I eat in the Bone Student Center?", "section": "dining", "phrases": ["bone student center", "mcalister", "qdoba", "timbers", "the landing"]},
    {"id": "D05", "category": "dining", "question": "Where can I find allergen information for dining hall food?", "section": "dining", "phrases": ["allergen", "netnutrition"]},
    {"id": "D06", "category": "dining", "question": "Can my parents eat in the dining halls for free?", "section": "dining", "phrases": ["parent"]},
    {"id": "D07", "category": "dining", "question": "Can I talk to a dietitian about my diet?", "section": "dining", "phrases": ["dietitian"]},
    {"id": "D08", "category": "dining", "question": "What meal plans are there for students who live off campus?", "section": "dining", "phrases": ["block", "off-campus", "off campus"]},

    # --- traps: nothing in the data answers these -------------------------------------------------
    {"id": "X01", "category": "trap", "question": "What are the prerequisites for IT 999?", "trap": True, "expect": "says IT 999 is not in the catalog"},
    {"id": "X02", "category": "trap", "question": "What is my current account balance?", "trap": True, "expect": "says it cannot see accounts; points to the billing portal"},
    {"id": "X03", "category": "trap", "question": "What room am I assigned to next semester?", "trap": True, "expect": "says it cannot see assignments; points to the housing portal"},
    {"id": "X04", "category": "trap", "question": "How much will the 7-Day Unlimited meal plan cost in 2031?", "trap": True, "expect": "does not invent a 2031 price"},
    {"id": "X05", "category": "trap", "question": "Who won the Redbirds football game last weekend?", "trap": True, "expect": "says this is outside what it knows"},
    {"id": "X06", "category": "trap", "question": "Tell me about Skyline Towers residence hall.", "trap": True, "expect": "says there is no such hall in its sources"},
]

print(f"{len(ARM1_QUESTIONS)} questions:",
      ", ".join(f"{n} {c}" for c, n in Counter(q["category"] for q in ARM1_QUESTIONS).items()))

In [ ]:
import contextlib
import csv
import io
import shutil

# Results go next to the cache: in Google Drive when it is mounted, so they survive the runtime.
ARM1_DIR     = (DRIVE_PATH or CACHE_DIR) / "experiments" / "arm1"
ANSWER_PAUSE = 6                          # seconds between Gemini calls - inside the free tier's per-minute limit
GRADES       = ("correct", "partial", "wrong")

# The host each website section was crawled from, to check an answer's links against.
SECTION_HOSTS = {section: {urlparse(seed).netloc for seed in seeds}
                 for section, seeds in WEB_SOURCES.items()}

# How the app's non-answers begin (matches_only() and answer_stream() in section 6).
FALLBACKS = {
    "**The model is busy": "busy",
    "**Gemini quota": "quota",
    "**Gemini rejected": "rejected",
    "**Closest matches": "no key",
    "Nothing in the catalog matched": "no matches",
}

# Wording that signals "I don't have that" - the automatic check for trap questions.
DECLINE_PHRASES = (
    "not find", "couldn't find", "could not find", "no information", "not able to", "unable to",
    "can't see", "cannot see", "can't access", "cannot access", "do not have access",
    "don't have access", "doesn't contain", "does not contain", "not in the", "no course",
    "isn't listed", "is not listed", "not listed", "doesn't appear", "does not appear",
    "don't see", "do not see", "not available", "not mentioned", "no mention", "not provided",
    "don't have", "do not have", "doesn't include", "does not include", "outside", "not include",
)

RESULT_FIELDS = [
    "id", "category", "question", "rule", "label_valid", "relevant_in_corpus",
    "routed_to", "route_ok", "first_relevant_rank", "hit5", "hit8", "rr", "top3", "top8_ids",
    "answer_status", "latency_s", "gemini_retries", "has_link", "cites_expected", "declined_auto",
    "answer", "grade", "notes",
]


def normalize_grade(value):
    """'Correct', 'c', 'yes' -> correct; 'partially correct', 'p' -> partial; 'incorrect', 'no' -> wrong."""
    v = str(value or "").strip().lower()
    if not v:
        return ""
    if v.startswith("part") or v in ("p", "0.5", "half"):
        return "partial"
    if v.startswith(("incorrect", "wrong", "not correct")) or v in ("w", "x", "n", "no", "0", "false"):
        return "wrong"
    if v.startswith(("correct", "right")) or v in ("c", "y", "yes", "1", "true"):
        return "correct"
    return ""


def norm_code(text):
    """'IT 214', 'it-214', 'IT214' -> 'IT214'."""
    return re.sub(r"[\s\-]", "", text or "").upper()


def describe_rule(q):
    if q.get("trap"):
        return "trap: " + q.get("expect", "should decline")
    if q.get("codes"):
        return "course " + " or ".join(q["codes"])
    return f"{q['section']} chunk containing: " + " / ".join(q["phrases"])


def is_relevant(q, doc):
    """Does this chunk satisfy the question's rule?"""
    meta = doc["metadata"]
    if q.get("codes"):
        return norm_code(meta.get("code")) in {norm_code(c) for c in q["codes"]}
    if q.get("phrases"):
        if meta.get("section", "courses") != q["section"]:
            return False
        if q["section"] == "courses":       # title + description, not the department header
            haystack = f"{meta.get('title', '')} {meta.get('description', '')}"
        else:
            haystack = doc["text"]
        haystack = haystack.lower()
        return any(p.lower() in haystack for p in q["phrases"])
    return False


def short_label(doc):
    """'IT 214 Social, Legal... [courses]' or 'FAQ / Refunds [billing]' - for the miss list."""
    meta = doc["metadata"]
    name = meta.get("title", "")
    if meta.get("heading"):
        name = f"{name} / {meta['heading']}"
    return f"{name[:60]} [{meta.get('section', 'courses')}]"


def first_relevant_rank(q, hits):
    for rank, hit in enumerate(hits, 1):
        if is_relevant(q, hit["doc"]):
            return rank
    return None


def answer_status(text):
    for start, label in FALLBACKS.items():
        if text.startswith(start):
            return label
    return "answered" if text.strip() else "empty"


def declined(text):
    low = text.lower().replace("’", "'")
    return any(phrase in low for phrase in DECLINE_PHRASES)


def cites_expected(q, text, hits):
    """Course questions: names an expected code. Website questions: links the expected site."""
    if q.get("section") in SECTION_HOSTS:
        hosts = SECTION_HOSTS[q["section"]]
        links = re.findall(r"https?://[^\s)\]>\"']+", text)
        return any(urlparse(link).netloc in hosts for link in links)
    if q.get("codes"):
        targets = q["codes"]
    else:                                   # topic: any relevant course the retriever found
        targets = [h["doc"]["metadata"]["code"] for h in hits if is_relevant(q, h["doc"])]
    compact = norm_code(text)
    return any(norm_code(code) in compact for code in targets if code)


def run_arm1(questions, ask_gemini=True, pause=ANSWER_PAUSE):
    """Score every question: retrieval always, Gemini's answer when ask_gemini is on."""
    corpus = APP.index.documents
    can_ask = ask_gemini and APP.answerer is not None
    live = can_ask and APP.answerer.available
    rows, quota_streak = [], 0

    for n, q in enumerate(questions, 1):
        trap = bool(q.get("trap"))
        hits = APP.index.search(q["question"], k=TOP_K)          # the exact call the chat makes
        in_corpus = None if trap else sum(1 for d in corpus if is_relevant(q, d))
        valid = trap or in_corpus > 0
        rank = first_relevant_rank(q, hits) if valid and not trap else None
        routed = sorted(route_sections(q["question"]))
        web = q.get("section") in SECTION_HOSTS
        scored = valid and not trap

        row = {
            "id": q["id"], "category": q["category"], "question": q["question"],
            "rule": describe_rule(q), "label_valid": valid,
            "relevant_in_corpus": "" if in_corpus is None else in_corpus,
            "routed_to": ", ".join(routed) or "-",
            "route_ok": (q["section"] in routed) if web else "",
            "first_relevant_rank": rank or "",
            "hit5": bool(rank and rank <= 5) if scored else "",
            "hit8": bool(rank and rank <= TOP_K) if scored else "",
            "rr": round(1 / rank, 4) if (scored and rank) else (0.0 if scored else ""),
            "top3": " | ".join(short_label(h["doc"]) for h in hits[:3]),
            "top8_ids": " ".join(h["doc"]["id"] for h in hits),
            "answer_status": "", "latency_s": "", "gemini_retries": "", "has_link": "",
            "cites_expected": "", "declined_auto": "", "answer": "", "grade": "", "notes": "",
        }

        if can_ask:
            log = io.StringIO()
            started = time.time()
            with contextlib.redirect_stdout(log):                # keep the retry chatter out of the table
                text, _ = APP.answerer.answer(q["question"], k=TOP_K)
            tries = [line for line in log.getvalue().splitlines() if line.startswith("[gemini]")]
            status = answer_status(text)
            row.update({
                "answer_status": status,
                "latency_s": round(time.time() - started, 2),
                "gemini_retries": len(tries),
                "has_link": "](http" in text,
                "cites_expected": cites_expected(q, text, hits) if (status == "answered" and scored) else "",
                "declined_auto": declined(text) if (status == "answered" and trap) else "",
                "answer": text,
            })
            for line in tries:
                print("      ", line[:120])
            quota_streak = quota_streak + 1 if status == "quota" else 0
            if quota_streak >= 3:
                print("   Gemini quota is used up - scoring the rest on retrieval only. Re-run later for answers.")
                can_ask = False
            elif live and n < len(questions):
                time.sleep(pause)

        rows.append(row)
        found = ("trap" if trap else "label matches nothing - left out" if not valid
                 else f"rank {rank}" if rank else "not in top 8")
        print(f"{q['id']:<4} {found:<32} {row['answer_status'] or '-':<10} {q['question'][:58]}")
    return rows


def _coerce(row):
    """Rows read back from CSV are all text; turn them back into numbers and booleans."""
    for field in ("label_valid", "route_ok", "hit5", "hit8", "has_link", "cites_expected", "declined_auto"):
        value = row.get(field, "")
        if isinstance(value, str):
            low = value.strip().lower()
            row[field] = True if low in ("true", "1", "yes") else False if low in ("false", "0", "no") else ""
    for field in ("relevant_in_corpus", "first_relevant_rank", "gemini_retries"):
        value = row.get(field, "")
        if isinstance(value, str):
            row[field] = int(float(value)) if value.strip() else ""
    for field in ("rr", "latency_s"):
        value = row.get(field, "")
        if isinstance(value, str):
            row[field] = float(value) if value.strip() else ""
    row["grade"] = normalize_grade(row.get("grade"))
    return row


def load_arm1_rows(path):
    with open(path, encoding="utf-8-sig", newline="") as f:
        return [_coerce(dict(row)) for row in csv.DictReader(f)]


def arm1_setup():
    dense = APP.index.collection is not None
    models = list(getattr(APP.answerer, "models", None) or [])
    return {
        "run_at": datetime.now().isoformat(timespec="seconds"),
        "catalog": APP.meta.get("catalog_name", ""),
        "data_checked_at": APP.meta.get("checked_at", ""),
        "chunks": len(APP.documents),
        "sections": dict(APP.sections),
        "embedding": EMBED_MODEL if dense else "TF-IDF fallback (MiniLM unavailable)",
        "vector_db": f"Chroma collection '{COLLECTION_NAME}', cosine distance" if dense else "none (TF-IDF fallback)",
        "top_k": TOP_K,
        "shown": SHOW_SOURCES,
        "generator": models[0] if models else "none (no GOOGLE_API_KEY)",
        "fallbacks": models[1:],
    }


def summarize_arm1(rows, setup=None):
    categories = list(dict.fromkeys(r["category"] for r in rows))
    scored = [r for r in rows if r["category"] != "trap" and r["label_valid"] is True]

    def block(group):
        n = len(group)
        return {"n": n,
                "hit5": sum(1 for r in group if r["hit5"] is True),
                "hit8": sum(1 for r in group if r["hit8"] is True),
                "mrr": round(sum(float(r["rr"] or 0) for r in group) / n, 3) if n else 0.0}

    retrieval = {c: block([r for r in scored if r["category"] == c]) for c in categories if c != "trap"}
    retrieval["all"] = block(scored)

    routed = [r for r in rows if r["route_ok"] in (True, False)]
    asked = [r for r in rows if r["answer_status"]]
    real = [r for r in asked if r["answer_status"] == "answered"]
    real_scored = [r for r in real if r["category"] != "trap" and r["cites_expected"] in (True, False)]
    real_web = [r for r in real if r["category"] in SECTION_HOSTS]
    real_traps = [r for r in real if r["category"] == "trap"]

    graded = [r for r in rows if r["grade"] in GRADES]
    manual = None
    if graded:
        manual = {c: {g: sum(1 for r in graded if r["category"] == c and r["grade"] == g) for g in GRADES}
                  for c in categories}
        manual["all"] = {g: sum(1 for r in graded if r["grade"] == g) for g in GRADES}

    return {
        "setup": setup or arm1_setup(),
        "categories": categories,
        "question_counts": dict(Counter(r["category"] for r in rows)),
        "excluded": [r["id"] for r in rows if r["label_valid"] is False],
        "retrieval": retrieval,
        "routing": {"ok": sum(1 for r in routed if r["route_ok"] is True), "of": len(routed)},
        "generation": {
            "asked": len(asked),
            "answered": len(real),
            "statuses": dict(Counter(r["answer_status"] for r in asked)),
            "cites": sum(1 for r in real_scored if r["cites_expected"] is True), "cites_of": len(real_scored),
            "links": sum(1 for r in real_web if r["has_link"] is True), "links_of": len(real_web),
            "declined": sum(1 for r in real_traps if r["declined_auto"] is True), "declined_of": len(real_traps),
            "avg_latency": round(sum(float(r["latency_s"] or 0) for r in asked) / len(asked), 1) if asked else 0.0,
            "retries": sum(int(r["gemini_retries"] or 0) for r in asked),
        },
        "manual": manual,
    }


def _frac(num, den):
    return f"{num}/{den} ({num / den:.0%})" if den else "n/a"


CATEGORY_LABELS = {"course code": "Course codes", "course topic": "Course topics", "billing": "Billing",
                   "housing": "Housing", "dining": "Dining", "trap": "Trap questions", "all": "**All scored**"}


def arm1_writeup(rows, summary):
    """The Google-doc section, written around this run's numbers. Edit it before you submit."""
    s, ret, gen, man = summary["setup"], summary["retrieval"], summary["generation"], summary["manual"]
    counts, label = summary["question_counts"], (lambda c: CATEGORY_LABELS.get(c, c.title()))
    scored_cats = [c for c in summary["categories"] if c != "trap"]
    out = []
    add = out.append

    add("## Experimental Arm 1: the assistant as built (baseline)")
    add("")
    add(f"*Run {s['run_at'].replace('T', ' at ')[:19]}. Every number below comes from "
        "`experiments/arm1/results.csv`, produced by section 13 of the notebook.*")
    add("")
    add("### Purpose")
    add("Arm 1 measures the ISU Campus Assistant exactly as it was built, before any changes. It is the "
        "baseline: each later arm changes one part of the system and is compared against these numbers on the "
        "same questions and the same saved data.")
    add("")
    add("### Hypothesis")
    add("*[Edit this to match what you expected before running.]* Course-code questions should be found almost "
        "every time, because a course code typed in a question is pinned to the top of the results. Topic "
        "questions should be harder, because they depend on matching meaning rather than exact words. Billing, "
        "housing and dining questions depend on the keyword router sending them to the right section. Trap "
        "questions, which have no answer in the data, should be declined rather than answered.")
    add("")
    add("### Setup")
    sections = ", ".join(f"{n:,} {name}" for name, n in sorted(s["sections"].items(), key=lambda kv: -kv[1]))
    chain = s["generator"] + (f" (fallbacks: {', '.join(s['fallbacks'])})" if s["fallbacks"] else "")
    add("| Component | Arm 1 setting |")
    add("|---|---|")
    add(f"| Data | {s['catalog'] or 'ISU catalog'}, crawled {s['data_checked_at'][:10] or 'unknown'}: {sections} ({s['chunks']:,} chunks) |")
    add("| Chunking | One chunk per course; web pages split on headings into ~1,000-character chunks with 150 characters of overlap |")
    add(f"| Embeddings | {s['embedding']} |")
    add(f"| Vector database | {s['vector_db']} |")
    add("| Retrieval | BM25 + vector search merged by reciprocal rank fusion (k = 60), course-code pin, keyword section router |")
    add(f"| Context | Top {s['top_k']} chunks sent to Gemini; {s['shown']} shown as source cards |")
    add(f"| Generator | {chain}, temperature 0.2 |")
    add(f"| Questions | {sum(counts.values())}: " + ", ".join(f"{n} {label(c).lower()}" for c, n in counts.items()) + " |")
    add("")
    add("### Method")
    add("Every question was asked through the same search and answer functions the chat interface uses. Each "
        "question has a rule for what counts as the right source: the expected course code, or a chunk from the "
        "expected section that contains a key phrase. Before scoring, every rule was checked against the loaded "
        "data" + ((f"; {len(summary['excluded'])} question" + (" was" if len(summary["excluded"]) == 1 else "s were")
                   + f" left out because the rule matched nothing ({', '.join(summary['excluded'])}).")
                  if summary["excluded"] else ", and every rule matched at least one chunk.")
        + " Trap questions have no right source and are judged only on whether the reply declines.")
    add("")
    add("Hit@5 counts the right source among the 5 source cards a user sees; Hit@8 among the 8 chunks given to "
        "Gemini; MRR averages 1 ÷ the rank of the first right source. The answer checks are automatic text "
        "matches: whether Gemini answered at all, whether the answer names the expected course or links the "
        "expected website, and whether a trap reply contains declining language.")
    add("")
    add("### Results: retrieval")
    add("| Category | Questions | Hit@5 | Hit@8 | MRR |")
    add("|---|---|---|---|---|")
    for c in scored_cats + ["all"]:
        b = ret.get(c)
        if not b:
            continue
        add(f"| {label(c)} | {b['n']} | {_frac(b['hit5'], b['n'])} | {_frac(b['hit8'], b['n'])} | {b['mrr']:.2f} |")
    add("")
    if summary["routing"]["of"]:
        add(f"The keyword router sent {_frac(summary['routing']['ok'], summary['routing']['of'])} of the billing, "
            "housing and dining questions to their own section.")
        add("")
    add("### Results: answers")
    if gen["asked"]:
        add("| Measure | Result |")
        add("|---|---|")
        add(f"| Gemini wrote an answer (no busy or quota fallback) | {_frac(gen['answered'], gen['asked'])} |")
        add(f"| Answer names the expected course or links the expected site | {_frac(gen['cites'], gen['cites_of'])} |")
        add(f"| Website answers that include a source link | {_frac(gen['links'], gen['links_of'])} |")
        add(f"| Trap questions declined (automatic check) | {_frac(gen['declined'], gen['declined_of'])} |")
        add(f"| Average time per question | {gen['avg_latency']} s |")
        add(f"| Gemini retries during the run | {gen['retries']} |")
        others = {k: v for k, v in gen["statuses"].items() if k != "answered"}
        if others:
            add("")
            add("Not answered: " + ", ".join(f"{v} {k}" for k, v in others.items()) + ".")
    else:
        add("*Gemini was not asked in this run (ASK_GEMINI was off), so only retrieval is reported.*")
    add("")
    add("**Manual grading**")
    if man:
        add("")
        add("| Category | Correct | Partial | Wrong |")
        add("|---|---|---|---|")
        for c in summary["categories"] + ["all"]:
            m = man.get(c)
            if m and sum(m.values()):
                total = sum(m.values())
                add(f"| {label(c)} | {_frac(m['correct'], total)} | {m['partial']} | {m['wrong']} |")
    else:
        add("*[Grade each answer in the `grade` column of results.csv as correct, partial or wrong, then run the "
            "grading cell to fill in this table.]*")
    add("")
    add("### What went wrong")
    misses = [r for r in rows if r["hit8"] is False]
    late = [r for r in rows if r["hit8"] is True and r["hit5"] is False]
    unanswered = [r for r in rows if r["answer_status"] and r["answer_status"] != "answered"]
    no_cite = [r for r in rows if r["cites_expected"] is False]
    trap_miss = [r for r in rows if r["declined_auto"] is False]
    if not (misses or late or unanswered or no_cite or trap_miss):
        add("Every scored question found its source among the visible source cards, and every automatic answer "
            "check passed.")
    for r in misses:
        add(f"- **{r['id']}** \"{r['question']}\": the right source was not in the top 8. It returned: {r['top3']}.")
    for r in late:
        add(f"- **{r['id']}** \"{r['question']}\": the right source was found at rank {r['first_relevant_rank']}, "
            "below the 5 visible source cards.")
    if no_cite:
        add(f"- {len(no_cite)} answer(s) did not name the expected course or link the expected site: "
            + ", ".join(r["id"] for r in no_cite) + ".")
    for r in trap_miss:
        add(f"- **{r['id']}** (trap) \"{r['question']}\": the reply did not clearly decline. Check whether it invented an answer.")
    if unanswered:
        add(f"- {len(unanswered)} question(s) got a fallback instead of an answer "
            f"({', '.join(r['id'] + ': ' + r['answer_status'] for r in unanswered)}).")
    add("")
    add("*[Explain the likely cause of each miss: different wording than the source, a page the crawler did not "
        "reach, the router sending the question elsewhere, or a chunk that split the answer.]*")
    add("")
    add("### Conclusion")
    total = ret["all"]
    if total["n"]:
        add(f"Arm 1 found the right source among the 8 chunks given to Gemini for {_frac(total['hit8'], total['n'])} "
            f"of the scored questions, and among the 5 source cards a user sees for "
            f"{_frac(total['hit5'], total['n'])}, with an MRR of {total['mrr']:.2f}.")
        rates = {c: ret[c]["hit8"] / ret[c]["n"] for c in scored_cats if ret.get(c, {}).get("n")}
        if len(rates) > 1:
            best, worst = max(rates, key=rates.get), min(rates, key=rates.get)
            if rates[best] != rates[worst]:
                add(f"The strongest category was {label(best).lower()} ({rates[best]:.0%} Hit@8) and the weakest "
                    f"was {label(worst).lower()} ({rates[worst]:.0%}).")
    add("*[Add your interpretation: what these numbers say about the design, and which result surprised you.]*")
    add("")
    add("Retrieval scores are repeatable for this saved data. Gemini's wording varies slightly between runs "
        "(temperature 0.2), so the answer checks can move by a question or two on a re-run, and the automatic "
        "checks are simple text matches; the manual grades are the authority on correctness.")
    add("")
    add("### Next: Arm 2")
    add("Arm 2 will change exactly one thing and keep these questions, this saved data and every other setting "
        "identical. *[Edit to match your plan - for example, keyword-only retrieval (BM25 alone, no vector "
        "search), which measures how much the vector database and rank fusion add.]*")
    add("")
    add("### Demo and files")
    add("- Demo: *[link to a screen recording or screenshots of the app answering Arm 1 questions]*")
    add("- Code, questions and results: *[your GitHub repo link]*`/experiments/arm1/`")
    return "\n".join(out)


def save_arm1(rows, summary, writeup):
    ARM1_DIR.mkdir(parents=True, exist_ok=True)
    with open(ARM1_DIR / "questions.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "category", "question", "rule"])
        for q in ARM1_QUESTIONS:
            writer.writerow([q["id"], q["category"], q["question"], describe_rule(q)])
    with open(ARM1_DIR / "results.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)
    (ARM1_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    (ARM1_DIR / "arm1_writeup.md").write_text(writeup, encoding="utf-8")
    print(f"\nsaved to {ARM1_DIR}: questions.csv, results.csv, summary.json, arm1_writeup.md")


def show_arm1(rows, writeup):
    """A per-question table, then the write-up rendered so it can be copied into a Google doc."""
    try:
        import pandas as pd
        from IPython.display import Markdown, display
    except ImportError:
        print(writeup)
        return
    columns = ["id", "category", "first_relevant_rank", "hit5", "hit8", "routed_to",
               "answer_status", "cites_expected", "declined_auto", "question"]
    display(pd.DataFrame(rows)[columns])
    display(Markdown(writeup))


def download_arm1():
    archive = shutil.make_archive(str(ARM1_DIR.parent / "arm1_results"), "zip", ARM1_DIR)
    try:
        from google.colab import files
        files.download(archive)
    except ImportError:
        print("zipped:", archive)


# --- grading -------------------------------------------------------------------

def arm1_rows_for_grading():
    """This session's results, or the ones an earlier run saved to the arm1 folder."""
    if globals().get("ARM1_ROWS"):
        return ARM1_ROWS
    saved = ARM1_DIR / "results.csv"
    if saved.exists():
        print(f"grading the saved run in {saved}")
        return load_arm1_rows(saved)
    print(f"No Arm 1 results yet - run the cell above with RUN_ARM1 ticked first (looked in {ARM1_DIR}).")
    return None


def finish_grading(rows):
    """Re-score with the grades now in `rows`; save everything and refresh the write-up."""
    global ARM1_ROWS, ARM1_SUMMARY, ARM1_WRITEUP
    previous = ARM1_DIR / "summary.json"
    setup = json.loads(previous.read_text())["setup"] if previous.exists() else None
    ARM1_ROWS = rows
    ARM1_SUMMARY = summarize_arm1(rows, setup=setup)
    ARM1_WRITEUP = arm1_writeup(rows, ARM1_SUMMARY)
    save_arm1(rows, ARM1_SUMMARY, ARM1_WRITEUP)
    counts = (ARM1_SUMMARY["manual"] or {}).get("all") or {g: 0 for g in GRADES}
    print(f"graded {sum(counts.values())} answers: " + ", ".join(f"{n} {g}" for g, n in counts.items()))
    return ARM1_SUMMARY


def grading_form(rows):
    """Grade inside the notebook: one dropdown per answer, then Save grades."""
    try:
        import ipywidgets as widgets
        from IPython.display import Markdown, display
    except ImportError:
        print("Widgets are not available here - choose the spreadsheet option instead.")
        return None

    gradable = [r for r in rows if r.get("answer")]
    if not gradable:
        print("These results have no Gemini answers to grade (ASK_GEMINI was off).")
        return None

    choices = [("not graded", ""), ("correct", "correct"), ("partial", "partial"), ("wrong", "wrong")]
    pickers, blocks = {}, []
    for r in gradable:
        head = (f"<b>{r['id']}</b> &middot; {html_lib.escape(r['category'])} &mdash; "
                f"{html_lib.escape(r['question'])}<br><span style='color:#666;font-size:12px'>"
                f"expected: {html_lib.escape(r['rule'])}</span>")
        body = ("<div style='white-space:pre-wrap;border-left:3px solid #CC0000;padding:4px 10px;"
                "margin:4px 0;max-height:200px;overflow:auto;font-size:13px'>"
                f"{html_lib.escape(r['answer'][:2000])}</div>")
        pickers[r["id"]] = widgets.Dropdown(options=choices, value=r.get("grade") or "", description="grade:")
        blocks.append(widgets.VBox([widgets.HTML(head + body), pickers[r["id"]]]))

    save = widgets.Button(description="Save grades", button_style="danger", icon="check")
    out = widgets.Output()

    def on_save(_):
        by_id = {r["id"]: r for r in rows}
        for qid, picker in pickers.items():
            by_id[qid]["grade"] = picker.value
        out.clear_output()
        with out:
            finish_grading(rows)
            display(Markdown(ARM1_WRITEUP))

    save.on_click(on_save)
    display(widgets.HTML(f"<b>{len(gradable)} answers to grade.</b> For trap questions, <i>correct</i> means it "
                         "declined or pointed to the right page without inventing anything. Press <b>Save grades</b> "
                         "as often as you like; grades are saved to the arm1 folder and come back next time."))
    display(widgets.VBox(blocks + [save, out]))
    return pickers, save


def read_grades(path):
    """{question id: grade} from a CSV or Excel file, whichever program saved it."""
    if path.suffix.lower() in (".xlsx", ".xls"):
        import pandas as pd
        records = pd.read_excel(path, dtype=str).fillna("").to_dict("records")
    else:
        raw = path.read_bytes()
        for encoding in ("utf-8-sig", "cp1252", "latin-1"):   # Sheets, Excel on Windows, anything else
            try:
                text = raw.decode(encoding)
                break
            except UnicodeDecodeError:
                continue
        records = list(csv.DictReader(io.StringIO(text)))
    grades = {}
    for record in records:
        record = {str(k).strip().lower(): v for k, v in record.items() if k is not None}
        qid = str(record.get("id") or "").strip().upper()
        grade = normalize_grade(record.get("grade"))
        if qid and grade:
            grades[qid] = grade
    return grades


def grade_from_file():
    """Grades from a spreadsheet saved in the arm1 folder - the newest CSV or Excel file that has any."""
    rows = arm1_rows_for_grading()
    if not rows:
        return None
    candidates = sorted((p for p in ARM1_DIR.glob("*")
                         if p.suffix.lower() in (".csv", ".xlsx", ".xls") and p.name != "questions.csv"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    found, source = {}, None
    for path in candidates:
        try:
            found = read_grades(path)
        except Exception as exc:
            print(f"skipped {path.name}: {type(exc).__name__}: {exc}")
            continue
        if found:
            source = path
            break
    if not found:
        print(f"No graded file found in {ARM1_DIR}. Save your graded copy of results.csv there "
              "(CSV or Excel, any name) with the grade column filled in, then run this again.")
        return None

    print(f"read {len(found)} grades from {source.name}")
    unknown = sorted(set(found) - {r["id"] for r in rows})
    if unknown:
        print("ignored ids that are not in these results:", ", ".join(unknown))
    for r in rows:
        if r["id"] in found:
            r["grade"] = found[r["id"]]
    summary = finish_grading(rows)
    show_arm1(rows, ARM1_WRITEUP)
    return summary

### Run Arm 1

Tick **RUN_ARM1** and run this cell. Leave **ASK_GEMINI** ticked for the full experiment; unticked, it scores
retrieval only, in seconds, without using any Gemini quota. When it finishes it shows a per-question table and
the write-up draft, and downloads `arm1_results.zip`.

In [ ]:
RUN_ARM1 = False  #@param {type:"boolean"}
ASK_GEMINI = True  #@param {type:"boolean"}

if not RUN_ARM1:
    print("Arm 1 skipped - tick RUN_ARM1 above and run this cell to score the assistant.")
elif "APP" not in globals() or not APP.documents:
    print("Start the assistant first (section 10) - Arm 1 scores the app that cell loads.")
elif "run_arm1" not in globals() or "ARM1_QUESTIONS" not in globals():
    print("Run the question-set and helper cells above first, then this one.")
else:
    ARM1_ROWS = run_arm1(ARM1_QUESTIONS, ask_gemini=ASK_GEMINI)
    ARM1_SUMMARY = summarize_arm1(ARM1_ROWS)
    ARM1_WRITEUP = arm1_writeup(ARM1_ROWS, ARM1_SUMMARY)
    save_arm1(ARM1_ROWS, ARM1_SUMMARY, ARM1_WRITEUP)
    show_arm1(ARM1_ROWS, ARM1_WRITEUP)
    download_arm1()

### Grade the answers (recommended)

The automatic checks catch missing sources and links, not wrong facts, so grade each answer by hand.
Tick **GRADE_ARM1** and run the cell below.

* **Form in this notebook** (default): every answer is listed with a grade dropdown. Choose *correct*,
  *partial* or *wrong* and press **Save grades**. You can grade in several sittings; saved grades come back
  next time, even after the runtime restarts.
* **Spreadsheet saved in the arm1 Drive folder**: grade `results.csv` in Google Sheets or Excel, then save it
  (CSV or Excel, any file name) into `MyDrive/isu_catalog_cache/experiments/arm1/` and run the cell.

For trap questions, *correct* means it declined or pointed to the right page without inventing anything.
Saving regenerates the write-up with a manual-grading table and updates every file in the arm1 folder.

In [ ]:
GRADE_ARM1 = False  #@param {type:"boolean"}
GRADE_WITH = "form in this notebook"  #@param ["form in this notebook", "spreadsheet saved in the arm1 Drive folder"]

if not GRADE_ARM1:
    print("Grading skipped - tick GRADE_ARM1 to grade the Arm 1 answers.")
elif GRADE_WITH.startswith("form"):
    graded_rows = arm1_rows_for_grading()
    if graded_rows:
        grading_form(graded_rows)
else:
    grade_from_file()